In [ ]:
#!pip install 
from huggingface_hub import login
login("")

In [1]:
!pip install ../RMSearch

Processing /workspace/RMSearch
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-4.57.0-py3-none-any.whl.metadata (41 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.6-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached vllm-0.10.1-cp38-abi3-manylinux1_x86_64.whl.metadata (15 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached sentence_transformers-5.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached ijson-3.4.0-cp312-cp312-m

# Process Data

In [ ]:
!pip install datasets
!pip install huggingface_hub

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

#dataset_name = "hatakeyama-llm-team/japanese2010"
#dataset_name = "hotchpotch/fineweb-2-edu-japanese-noise-detect-raw"
dataset_name = "HuggingFaceTB/smollm-corpus"
save_name = "smollm-corpus"

ds = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train", num_proc=16)
#ds = load_dataset(dataset_name)
ds

In [ ]:
from datasets import load_dataset, Features, Value

features = Features({
    "index": Value("int64"),   # extra column that is really there
    "text":  Value("string")
})

ds = load_dataset(
    "hatakeyama-llm-team/japanese2010",
    split="train",             # or any split you need
    features=features,         # override the script‑defined schema
    #ignore_verifications=True  # skip size/sha checks – speeds things up
)                              # :contentReference[oaicite:0]{index=0}

# drop the column you don’t need
ds = ds.remove_columns("index")


In [ ]:
n_sample_train = 100000
#save_name = "ja-web-text-2"
save_name = "smollm-corpus"

ds = ds.shuffle(seed=42)
ds = ds.select(range(n_sample_train))

import pandas as pd   # only needed once
from datasets import DatasetDict  # your DatasetDict is already in memory as `ds`

df = ds.to_pandas()          # convert the split to a pandas DataFrame
print(df.shape)                 # (108000, 15) in your example
print(df.head())                 # (108000, 15) in your example
print(ds)

#ds = ds.shuffle(seed=42)
#ds = ds.select(range(n_sample))
ds.save_to_disk(f"./data/{save_name}")
df.to_csv(f"./data/{save_name}/df.csv", index=False)

In [ ]:
df = ds.to_pandas()          # convert the split to a pandas DataFrame
print(df.shape)                 # (108000, 15) in your example
print(df.head())                 # (108000, 15) in your example
print(ds)

#ds = ds.shuffle(seed=42)
#ds = ds.select(range(n_sample))
ds.save_to_disk(f"./data/{save_name}")
df.to_csv(f"./data/{save_name}/df.csv", index=False)

In [ ]:
n_sample_train = 100000
n_sample_test = 8000
#save_name = "ja-web-text-2"
save_name = "smollm-corpus"

ds = ds.shuffle(seed=42)

ds["train"] = ds["train"].select(range(n_sample_train))
ds["test"] = ds["test"].select(range(n_sample_test))

import pandas as pd   # only needed once
from datasets import DatasetDict  # your DatasetDict is already in memory as `ds`

def datasetdict_to_pandas(dataset_dict: DatasetDict) -> pd.DataFrame:
    """Merge all splits in a DatasetDict into a single DataFrame with a `split` column."""
    frames = []
    for split_name, split_ds in dataset_dict.items():
        df = split_ds.to_pandas()          # convert the split to a pandas DataFrame
        df["split"] = split_name           # label rows with their origin
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

df_all = datasetdict_to_pandas(ds)  # ds is your original DatasetDict
print(df_all.shape)                 # (108000, 15) in your example
print(df_all.head())                 # (108000, 15) in your example

#ds = ds.shuffle(seed=42)
#ds = ds.select(range(n_sample))
ds.save_to_disk(f"./data/{save_name}")
print(ds)
df_all.to_csv(f"./data/{save_name}/df.csv", index=False)

In [ ]:
aijsan

In [ ]:
# Make small df
n_small_sample = 10000
import pandas as pd

save_name = "smollm-corpus"

df_all = pd.read_csv(f"./data/{save_name}/df.csv")
df = df_all.sample(n_small_sample)
df.to_csv(f"./data/{save_name}/df_small.csv", index=False)
df

# Generate Tag Graph

In [ ]:
!pip install -U vllm

In [ ]:
import pandas as pd      # only needed if you want to manipulate the result further
from typing import List, Tuple, Dict, Any, Optional
from generate_tag_graph2 import generate_tag, embed_tags, get_tag_group, generate_representative_tag, _embed_pool_context, generate_tag_tree
import os, json, random
import torch

"""
Minimal debugging run. Adjust model names to ones you have locally.
- For vLLM (generation): use a small instruct model, e.g. "Qwen2.5-3B-Instruct" or "meta-llama/Meta-Llama-3-8B-Instruct".
- For embeddings: the user’s reference model "intfloat/e5-mistral-7b-instruct" works with SentenceTransformer.
"""
# ----- CONFIG -----
working_dir = "/workspace/RMS_exp"
data_name = "smollm-corpus"
#data_name = "test"
GEN_MODEL = os.environ.get("VLLM_MODEL", "/workspace/qwen7b")
EMB_MODEL = os.environ.get("EMB_MODEL", "intfloat/e5-mistral-7b-instruct")
SAVE_EMB = f"{working_dir}/data/{data_name}/key_embeddings.pt"
N_GROUP = 5000
N_TAG_SAMPLE = 6
tensor_parallel_size = 1
num_instances = 1
device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)
    
random.seed(0)

'''
# ----- SAMPLE KEYS (replace with yours) -----
sample_keys = [
    "Graph-based retrieval augmentation for enterprise documents",
    "Reward models for search result re-ranking",
    "Neural sparse indexing for web-scale retrieval",
    "Semantic tagging of legal contracts",
    "LLM orchestration with multi-agent planners",
    "Efficient RAG with vector + keyword hybrid",
    "Biomedical literature triage with MeSH terms",
    "GPU-efficient vLLM serving on multi-GPU",
    "Evaluation of negotiation agents with self-play",
    "Knowledge graph construction from PDFs",
]
'''

csv_name = "df_small.csv"
#ds = load_from_disk(f"./data/{save_name}") 
#df = ds.to_pandas()
df = pd.read_csv(f"./data/{data_name}/{csv_name}")
keys = df['text'].to_list()

if not os.path.exists(f"/workspace/RMS_exp/data/{data_name}"):
    os.makedirs(f"/workspace/RMS_exp/data/{data_name}")

In [ ]:
print(">>> Generating tags with vLLM ...")
tag_recs = generate_tag(keys, model_name=GEN_MODEL, batch_size=1000)
for r in tag_recs[:2]:
    print(f"[key_id={r['key_id']}] {r['key']}\n  tags={r['tags']}")

with open(f"{working_dir}/data/{data_name}/tag_recs.json", "w") as f:
    json.dump(tag_recs, f)

In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)

print("\n>>> Embedding tags with SentenceTransformer ...")

pool_settings = {
    "tensor_parallel_size": tensor_parallel_size,
    "num_instances": num_instances,
    "device_groups": device_groups,
    #"llm_kwargs": llm_kwargs,
}
batch_size = 10
timeout = 1000

with _embed_pool_context(EMB_MODEL, **pool_settings) as pool:
    emb = embed_tags(
        tag_records=tag_recs,
        embed_model_name=EMB_MODEL,
        pool=pool,
        worker_batch_size=batch_size,
        timeout_s=timeout,
    )

print("embeddings:", tuple(emb.shape), emb.dtype, emb.device)

torch.save(emb, f"{working_dir}/data/{data_name}/key_embeddings.pt")

In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)
emb = torch.load(f"{working_dir}/data/{data_name}/key_embeddings.pt")

print("\n>>> Clustering keys into groups ...")
tag_recs, centroids, group_recs = get_tag_group(tag_records=tag_recs, embeddings=emb, n_group=N_GROUP)
for g in group_recs[:10]:
    print(f"[group {g['group_id']}] members≈{len(g['tag_ids'])} tags_in_union={len(g['tags'])}")

torch.save(centroids, f"{working_dir}/data/{data_name}/centroids.pt")
with open(f"{working_dir}/data/{data_name}/tag_recs.json", "w") as f:
    json.dump(tag_recs, f)
with open(f"{working_dir}/data/{data_name}/group_recs.json", "w") as f:
    json.dump(group_recs, f)



In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)
with open(f"{working_dir}/data/{data_name}/group_recs.json") as f:
    group_recs = json.load(f)

    
print("\n>>> Generating representative tag per group with vLLM ...")
group_recs = generate_representative_tag(
    tag_records=tag_recs,
    group_records=group_recs,
    #embeddings=emb,
    n_tag_sample=N_TAG_SAMPLE,
    model_name=GEN_MODEL
)
for g in group_recs[:10]:
    print(f"[group {g['group_id']}] rep='{g['representative_tag']}'")

with open(f"{working_dir}/data/{data_name}/group_recs.json", "w") as f:
    json.dump(group_recs, f)

print("\n>>> Done. Saved reuslt to:", f"{working_dir}/data/{data_name}")


In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)
with open(f"{working_dir}/data/{data_name}/group_recs.json") as f:
    group_recs = json.load(f)
centroids = torch.load(f"{working_dir}/data/{data_name}/centroids.pt")

tag_tree_recs = generate_tag_tree(
    group_records = group_recs,
    centroids = centroids,
    tree_struc = [500, 50, 10],
    n_tag_sample = 20,
    model_name=GEN_MODEL,
    tensor_parallel_size=tensor_parallel_size,
    num_instances=num_instances,
    worker_batch_size=100,
)

with open(f"{working_dir}/data/{data_name}/tag_tree_recs.json", "w") as f:
    json.dump(tag_tree_recs, f)

In [ ]:
def generate_tag(keys: List[str], model_name: str):
    return tag_records # [{"key":, "key_id":, "tags":[]}, ...]

def embed_tags(tag_records, save_path, embed_model_name:str):
    return embeddings # torch.tensor([[...], ...]). # shape:[n_keys, embed_dim]

def get_tag_group(tag_records, embeddings, n_group:int):

    # tag_records = [{"key":str, "key_id":int, "tags":List[str], "group_ids":List[int]}, ...]
    # centroids = torch.tensor([[...], ...])   # shape: [n_group, embed_dim]
    # group_records = [{"group_id":int, "tags":List[str], "tag_ids":List[Tuple[int, int]]}]. # tag_ids = (key_id where the tag comes from, tag_id in the tags in dict in tag_records)
    return tag_records, centroids, group_records

def generate_representative_tag(tag_records, group_records, embeddings, n_tag_sample:int, model_name: str):
    # get n_tag_sample tags sampled from each group tags and llm generates a longer tag which represents the group.
    
    # group_records = [{"group_id":int, "representative_tag":str, "centroid":torch.tensor([]), "tags":List[str], "tag_ids":List[Tuple[int, int]]}]
    return group_records

def generate_tag_tree(group_records, centroids, tree_struc: List[int], n_tag_sample:int):
    # ex. 
    # tree_struc=[500, 50, 10] 
    # len(centroids)=5000 
    # n_tag_sample=20

    # 1. apply k_means to centroids and get 500 new_centroids
    # 2. from representative tags in each group, make new representative tag
    # 3. iterate this until the end of tree struc and get tag_tree
    # tag_tree_recs = [{"tag":, "children": [{"tag":, "children":...}, ...]}, ...]
    # here each tag is same as representative tag 
    # tag_tree_recs represents tree structure of tags and there are 10 tags at the top, 50 in total at second depth, and ...
    # each tag branch should be children of tag which is generated from it

    return tag_tree_recs


# Generate Tag Graph2 (Using outsource library for hierarchical kmeans)

In [ ]:
!pip install -U vllm

In [ ]:
import pandas as pd      # only needed if you want to manipulate the result further
from typing import List, Tuple, Dict, Any, Optional
from generate_tag_graph2 import generate_tag, embed_tags, get_tag_group, generate_representative_tag, _embed_pool_context, generate_tag_tree
import os, json, random, re
import torch
from hierarchical_kmeans import HierarchicalKMeans

"""
Minimal debugging run. Adjust model names to ones you have locally.
- For vLLM (generation): use a small instruct model, e.g. "Qwen2.5-3B-Instruct" or "meta-llama/Meta-Llama-3-8B-Instruct".
- For embeddings: the user’s reference model "intfloat/e5-mistral-7b-instruct" works with SentenceTransformer.
"""
# ----- CONFIG -----
working_dir = "/workspace/RMS_exp"
data_name = "smollm-corpus"
#data_name = "test"
GEN_MODEL = os.environ.get("VLLM_MODEL", "/workspace/qwen7b")
EMB_MODEL = os.environ.get("EMB_MODEL", "intfloat/e5-mistral-7b-instruct")
SAVE_EMB = f"{working_dir}/data/{data_name}/key_embeddings.pt"
N_GROUP = 5000
N_TAG_SAMPLE = 6
tensor_parallel_size = 1
num_instances = 1
device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)
    
random.seed(0)

'''
# ----- SAMPLE KEYS (replace with yours) -----
sample_keys = [
    "Graph-based retrieval augmentation for enterprise documents",
    "Reward models for search result re-ranking",
    "Neural sparse indexing for web-scale retrieval",
    "Semantic tagging of legal contracts",
    "LLM orchestration with multi-agent planners",
    "Efficient RAG with vector + keyword hybrid",
    "Biomedical literature triage with MeSH terms",
    "GPU-efficient vLLM serving on multi-GPU",
    "Evaluation of negotiation agents with self-play",
    "Knowledge graph construction from PDFs",
]
'''

csv_name = "df_small.csv"
#ds = load_from_disk(f"./data/{save_name}") 
#df = ds.to_pandas()
df = pd.read_csv(f"./data/{data_name}/{csv_name}")
keys = df['text'].to_list()

if not os.path.exists(f"/workspace/RMS_exp/data/{data_name}"):
    os.makedirs(f"/workspace/RMS_exp/data/{data_name}")

In [ ]:
print(">>> Generating tags with vLLM ...")
tag_recs = generate_tag(keys, model_name=GEN_MODEL, batch_size=1000)
for r in tag_recs[:2]:
    print(f"[key_id={r['key_id']}] {r['key']}\n  tags={r['tags']}")

with open(f"{working_dir}/data/{data_name}/tag_recs.json", "w") as f:
    json.dump(tag_recs, f)

In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)

print("\n>>> Embedding tags with SentenceTransformer ...")

pool_settings = {
    "tensor_parallel_size": tensor_parallel_size,
    "num_instances": num_instances,
    "device_groups": device_groups,
    #"llm_kwargs": llm_kwargs,
}
batch_size = 10
timeout = 1000

with _embed_pool_context(EMB_MODEL, **pool_settings) as pool:
    emb, tag_meta = embed_tags(
        tag_records=tag_recs,
        embed_model_name=EMB_MODEL,
        pool=pool,
        worker_batch_size=batch_size,
        timeout_s=timeout,
    )

print("embeddings:", tuple(emb.shape), emb.dtype, emb.device)

# tag_emb = torch.tensor([[...], ...])
# tag_meta = [(key_id, tag_id), ...]

torch.save(emb, f"{working_dir}/data/{data_name}/key_embeddings.pt")
with open(f"{working_dir}/data/{data_name}/tag_meta.json", "w") as f:
    json.dump(tag_meta, f)

In [ ]:
with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)

tag_meta = []
for key_id, tag_dict in enumerate(tag_recs):
    for tag_id, tag in enumerate(tag_dict["tags"]):
        tag_meta.append((key_id, tag_id))

with open(f"{working_dir}/data/{data_name}/tag_meta.json", "w") as f:
    json.dump(tag_meta, f)

In [ ]:
emb = torch.load(f"{working_dir}/data/{data_name}/key_embeddings.pt").detach().cpu()
with open(f"{working_dir}/data/{data_name}/tag_meta.json") as f:
    tag_meta = json.load(f)

# Build a k-means tree with branching factor 3, and leaf size < 60
hkm = HierarchicalKMeans(n_clusters=10, max_leaf_size=60, random_state=0)
hkm.fit(emb)

# Show leaf sizes
members = hkm.leaf_members()
print("Number of leaves:", len(members))
sizes = {nid: idxs for nid, idxs in members.items()}
#print("Leaf sizes:", sizes)

'''
# Predict which leaf a new point belongs to
test_pts = np.array([[0.2, 0.1], [5.2, 5.0], [0.1, 5.9]])
print("Assignments:", hkm.predict(test_pts))
'''


In [ ]:
from typing import Dict, Any, List, Tuple, Iterable

def _as_int_list(xs: Iterable) -> List[int]:
    # Works for numpy arrays, lists, tuples—anything iterable of numbers.
    return [int(x) for x in xs]

def _sorted_keys_numeric(keys: Iterable[str]) -> List[str]:
    # Sort child segment keys numerically if possible (e.g., '0','1','2',...)
    return sorted(keys, key=lambda k: (0, int(k)) if k.isdigit() else (1, k))

def convert_tree_dict_to_json(tree: Dict[str, Iterable]) -> List[Dict[str, Any]]:
    """
    Convert a dict like {'root/0/1': np.array([...]), 'root/1': np.array([...]), ...}
    into a JSON-like nested structure:
      [
        {"tag_ids": [], "children": [ {"tag_ids":[...]}, {"tag_ids":[...]} ]},
        {"tag_ids":[...]}
      ]
    """
    # Internal node storage by path tuple (excluding the literal 'root')
    Node = Dict[str, Any]
    nodes: Dict[Tuple[str, ...], Node] = {}

    def get_node(path: Tuple[str, ...]) -> Node:
        if path not in nodes:
            nodes[path] = {"tag_ids": [], "children": {}}
        return nodes[path]

    # Ensure root exists
    get_node(())  # path () represents 'root' itself

    # Insert all leaves; create intermediates as needed
    for key, ids in tree.items():
        parts = key.split('/')
        if not parts or parts[0] != 'root':
            raise ValueError(f"All keys must start with 'root': got {key}")
        segs = tuple(parts[1:])  # e.g., ('0','1') or ('1',)

        # Create intermediate nodes
        for i in range(len(segs)):
            parent_path = segs[:i]
            child_key = segs[i]
            parent = get_node(parent_path)
            child_path = segs[:i+1]
            child = get_node(child_path)
            parent["children"].setdefault(child_key, child)

        # Set tag_ids on the leaf node
        leaf = get_node(segs)
        leaf["tag_ids"] = _as_int_list(ids)

    # Recursively serialize node -> {"tag_ids": [...], "children":[...]} dropping empty children
    def serialize(node: Node) -> Dict[str, Any]:
        out = {"tag_ids": node["tag_ids"]}
        if node["children"]:
            children_objs = []
            for k in _sorted_keys_numeric(node["children"].keys()):
                child = node["children"][k]
                children_objs.append(serialize(child))
            out["children"] = children_objs
        return out

    # Top-level output is the list of root's immediate children
    root = nodes[()]
    result = []
    for k in _sorted_keys_numeric(root["children"].keys()):
        result.append(serialize(root["children"][k]))
    return result

tag_tree_recs = convert_tree_dict_to_json(sizes)
save_path = f"{working_dir}/data/{data_name}/tag_tree_recs.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(tag_tree_recs, f, ensure_ascii=False, indent=2)


In [ ]:
# Use Model from local environment

import asyncio
from transformers import AutoTokenizer
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams
import time, warnings
import os
from tqdm import tqdm

model_name = "/workspace/qwen7b"
tensor_parallel_size = 1
pipeline_parallel_size = 1
data_parallel_size = 1

engine_args = AsyncEngineArgs(
    model = model_name,
    tensor_parallel_size = tensor_parallel_size,
    pipeline_parallel_size = pipeline_parallel_size,
    #data_parallel_size = data_parallel_size,
    gpu_memory_utilization=0.95,
    disable_log_stats=True,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)
engine.log_requests = False
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

from jsonschema import validate, ValidationError

def validate_item(item, schema):
    try:
        validate(item, schema)
        return True, None
    except ValidationError as e:
        return False, str(e)


class AllRequests:
    
    def __init__(self, max_request):
        self.max_request = max_request
        self.requests = []
        self.request_ids = []
        self.request_id = 0
        self.results = []
        self.finished_ids = []
        self.progress_bar = None
        
    def add(self, request):
        self.requests.append(request)
        self.request_ids.append(self.request_id)
        self.request_id += 1
    
    async def process(self, model=model_name, max_tokens = 3000, temperature=0.4, save_dir = "progress_log", restart = False):

        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        if restart:
            if os.path.exists(f"{save_dir}/finished_ids.json") and os.path.exists(f"{save_dir}/results.json"):
                with open(f"{save_dir}/finished_ids.json") as f:
                    finished_ids = json.load(f)
                with open(f"{save_dir}/results.json") as f:
                    self.results = json.load(f)
                for finished_id in finished_ids:
                    self.finished_ids.append(finished_id)
                    id = self.request_ids.index(finished_id)
                    self.request_ids.pop(id)
                    self.requests.pop(id)
        else:
            if os.path.exists(f"{save_dir}/finished_ids.json"):
                os.remove(f"{save_dir}/finished_ids.json")
            if os.path.exists(f"{save_dir}/results.json"):
                os.remove(f"{save_dir}/results.json")

        total = len(self.requests)
        self.progress_bar = tqdm(total=total, desc="Processing Requests")

        await asyncio.gather(
            *[self.process_requests(temperature = temperature, max_tokens = max_tokens, save_dir=save_dir) for _ in range(self.max_request)]
        )

        self.progress_bar.close()
            
        return self.results


    async def process_requests(self, max_tokens = 3000, temperature=0.4, save_dir = "progress_log"):

        while len(self.requests) != 0:
            request_dict = self.requests.pop(0)
            request_id = self.request_ids.pop(0)

            prompt = request_dict["prompt"]

            final_output = None
            results_generator = engine.generate(prompt, SamplingParams(temperature=temperature, max_tokens=max_tokens), request_id)
            async for request_output in results_generator:
                # print(request_output) => for streaming
                final_output = request_output

            output = final_output.outputs[0].text
            
            request_dict["output"] = output

            valid = False
            try:
                json.loads(json.dumps(request_dict))
                valid = True
            except Exception as e:
                print("\nrequest id : ", request_id)
                print(e)
            #valid, error = validate_item(request_dict, schema)

            if valid:
                self.results.append(request_dict)
                self.finished_ids.append(request_id)
    
                with open(f"{save_dir}/results.json", "w") as f:
                    json.dump(self.results, f)
                with open(f"{save_dir}/finished_ids.json", "w") as f:
                    json.dump(self.finished_ids, f)
    
            self.progress_bar.update(1)
            

In [ ]:
import json
import os
import random
from typing import Any, Dict, List, Optional

with open(f"{working_dir}/data/{data_name}/tag_recs.json") as f:
    tag_recs = json.load(f)
with open(f"{working_dir}/data/{data_name}/tag_tree_recs.json") as f:
    tag_tree_recs = json.load(f)
with open(f"{working_dir}/data/{data_name}/tag_meta.json") as f:
    tag_meta = json.load(f)

# tag_recs = [{"key":, "key_id":, "tags":[]}, ...]
# tag_tree_recs = [{"tag_ids":, "children":[{"tag_ids":, "children":[...]}]}, ...]
# tag_meta = [(key_id, tag_id), ...]    # each tuple is at emb_tag_id. (len(key_embeddings) = len(tag_meta))


'''
# -----------------------------
# Dummy inputs for testing
# -----------------------------
# Minimal toy tree shaped like your converted structure:
tag_tree_recs = [
    {
        "tag_ids": [],  # internal node
        "children": [
            {"tag_ids": [202, 230], "tags": ["apple", "red apple"]},  # leaf
            {"tag_ids": [257, 275], "tags": ["banana", "ripe banana"]},  # leaf
        ],
    },
    {
        "tag_ids": [400, 403],  # leaf at top-level
        "tags": ["general-tools", "misc"]
    },
]

# tag_recs maps integer id -> {"tag": "..."}.
# Only needed by your make_leaf_tag_recs; we keep it for completeness.
tag_recs = {
    202: {"tag": "apple"},
    230: {"tag": "red apple"},
    257: {"tag": "banana"},
    275: {"tag": "ripe banana"},
    400: {"tag": "general-tools"},
    403: {"tag": "misc"},
}
'''

# -----------------------------
# Utilities & dummies
# -----------------------------
n_tag_sample = 10
random.seed(42)

'''
class AllRequests:
    """A dummy collector that returns a deterministic 'representative' tag for each request."""
    def __init__(self, max_request: int = 50):
        self.max_request = max_request
        self.reqs: List[Dict[str, Any]] = []

    def add(self, request: Dict[str, Any]):
        if len(self.reqs) >= self.max_request:
            # For a real system, you might batch; here just append up to max
            return
        self.reqs.append(request)

    def process(self, max_tokens: int = 3000, temperature: float = 0.0,
                save_dir: Optional[str] = None, restart: bool = True) -> List[Dict[str, Any]]:
        # A deterministic "LLM" that creates a short representative tag from the prompt
        os.makedirs(save_dir or "progress_questions", exist_ok=True)

        results = []
        for i, req in enumerate(self.reqs):
            # Super simple heuristic: take the first bullet after "Sample tags:"
            lines = [ln.strip("- ").strip() for ln in req["prompt"].splitlines() if ln.strip().startswith("- ")]
            candidate = lines[0] if lines else "general"
            # Shorten to <= 6 words as your prompt asks
            rep = " ".join(candidate.split()[:6]) or "general"
            results.append({"tag_ids": req["tag_ids"], "output": rep})
        # Reset if restart requested
        if restart:
            self.reqs = []
        return results
'''

# -----------------------------
# Helpers to populate leaf "tags"
# (kept but made robust: it won’t crash if tag_id missing in tag_recs)
# -----------------------------
def make_leaf_tag_recs(tag_tree_recs, tag_meta, tag_recs):
    def walk(node_list: List[Dict[str, Any]]):
        for tag_dict in node_list:
            if tag_dict.get("tag_ids"):
                tags = []
                for emb_tag_id in tag_dict["tag_ids"]:
                    key_id, tag_id = tag_meta[emb_tag_id]
                    tags.append(tag_recs[key_id]["tags"][tag_id])

                if tags:
                    tag_dict["tags"] = tags
            if "children" in tag_dict:
                walk(tag_dict["children"])
    walk(tag_tree_recs)
    return tag_tree_recs


tag_tree_recs = make_leaf_tag_recs(tag_tree_recs, tag_meta, tag_recs)

# -----------------------------
# Helpers to navigate the tree by index path
# -----------------------------
def get_node_by_path(root_list: List[Dict[str, Any]], path: List[int]) -> Dict[str, Any]:
    cur_list = root_list
    node = None
    for idx in path:
        node = cur_list[idx]
        cur_list = node.get("children", [])
    # If path empty, there's no "node" object corresponding to the root-list container.
    if path == []:
        # Represent the root container as a synthetic node; we won’t assign a tag to it.
        return {"_synthetic_root": True, "children": root_list}
    return node

def set_representative_tag(tag_tree_recs, path: List[int], representative_tag: str):
    if not path:
        # Avoid setting tag on the synthetic "root" container
        raise ValueError("path should point to a real node (non-empty path)")
    node = get_node_by_path(tag_tree_recs, path)
    node["tag"] = representative_tag
    return tag_tree_recs

def is_leaf(node: Dict[str, Any]) -> bool:
    return "children" not in node or not node["children"]

def extract_text(textC: str, textB: str) -> str | None:
    # Build a non-greedy pattern like r'<tag>(.*?)</tag>'
    pattern = rf"<{re.escape(textB)}>(.*?)</{re.escape(textB)}>"
    match = re.search(pattern, textC, flags=re.DOTALL)
    return match.group(1) if match else None

# -----------------------------
# Core: request generation and iterative labeling
# -----------------------------
async def get_representative_tag_request(tag_tree_recs):
    """
    One pass:
      - Ensure all leaves have a 'tag' (derived from their 'tags' if necessary).
      - For each internal node whose children ALL have 'tag' and the node itself lacks 'tag',
        enqueue a request with children's tags.
      - If no requests were created, return (tag_tree_recs, True) meaning 'done'.
      - Otherwise, process and set the new tags, return (tag_tree_recs, False) to continue.
    """
    all_requests = AllRequests(max_request=10)
    progress_made = False

    def ensure_leaf_tags(node: Dict[str, Any]):
        nonlocal progress_made
        if is_leaf(node):
            if "tag" not in node:
                # derive a simple representative from its own tags (if any)
                tags = node.get("tags", [])
                if tags:
                    node["tag"] = tags[0]
                else:
                    node["tag"] = "general"
                progress_made = True
        else:
            for ch in node["children"]:
                ensure_leaf_tags(ch)

    def enqueue_when_children_tagged(node_list: List[Dict[str, Any]], path: List[int]):
        """
        Consider the node represented by 'path' (its children are node_list).
        If all children of this node have 'tag' and the node itself lacks it,
        enqueue a request to generate it.
        """

        # First recurse into children to finish bottom-up
        for i, ch in enumerate(node_list):
            if not is_leaf(ch):
                enqueue_when_children_tagged(ch["children"], path + [i])

        # there must be tag_ids inside node_list 
        # after getting tag_ids -> get tag from tag_recs, tag_meta
        # see /Users/multivac/github_project/RMSearch/images/example_of_tag_tree_recs.png


        # Now check this parent node itself (skip synthetic root: path==[])
        # Collect immediate children tags:

        all_children_have_tag = all(("tag" in ch) for ch in node_list)
        if path != [] and all_children_have_tag:
            parent_node = get_node_by_path(tag_tree_recs, path)
            if "tag" not in parent_node:
                # Build sample from children's tags
                child_tags = [ch["tag"] for ch in node_list]
                sample = ["general"] if not child_tags else (
                    child_tags if len(child_tags) <= n_tag_sample else random.sample(child_tags, n_tag_sample)
                )
                lines = "\n".join(f"{i+1}. {t}" for i, t in enumerate(sample))
                '''
                prompt = (
                    "You are a taxonomy expert. Given the following sample tags from one cluster,\n"
                    "produce ONE concise representative tag (≤ 6 words) that best describes them all.\n"
                    "Do NOT include punctuation at the end. Output ONLY the tag text, nothing else.\n\n"
                    f"Sample tags:\n{lines}\n\nRepresentative tag:"
                )
                '''
                prompt = (
                    "You are a taxonomy expert. Given the following sample tags from one cluster,\n"
                    f"Sample tags:\n{lines}\n\n"
                    "Produce ONE concise representative tag (less than 10 words) that best describes them all.\n"
                    "Please enclose the representative tag by <tag></tag>."
                )
                all_requests.add({"path": path, "prompt": prompt})

    # 1) Ensure all leaves have 'tag'
    # Walk from a synthetic root node so we can reuse the traversal
    synth_root = {"children": tag_tree_recs}
    ensure_leaf_tags(synth_root)

    # 2) For each internal node whose children have tags but node lacks tag, enqueue
    enqueue_when_children_tagged(tag_tree_recs, [])

    # If nothing to request and we made no progress at leaves, we're done
    if not all_requests.requests:
        return tag_tree_recs, True

    # 3) Process requests (dummy) and set tags on those internal nodes
    results = await all_requests.process(max_tokens=3000, temperature=0.0,
                                   save_dir="progress_questions", restart=False)

    for result in results:
        path = result["path"]
        output = result["output"]
        tag = extract_text(output, "tag")
        set_representative_tag(tag_tree_recs, path, tag)

    return tag_tree_recs, False

# -----------------------------
# Driver loop (iterative until convergence)
# -----------------------------
async def build_representative_tags(tag_tree_recs, save_path=None):
    while_end = False
    # Optionally, call make_leaf_tag_recs first if you only have tag_ids:
    # tag_tree_recs = make_leaf_tag_recs(tag_tree_recs, tag_recs)

    # In this demo, leaves already have "tags", so we skip.
    while not while_end:
        tag_tree_recs, while_end = await get_representative_tag_request(tag_tree_recs)

    # Save
    if save_path:
        if not os.path.exists(os.path.dirname(save_path)):
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(tag_tree_recs, f, ensure_ascii=False, indent=2)
    else:
        os.makedirs("data/demo_dir", exist_ok=True)
        with open("data/demo_dir/tag_tree_recs.json", "w", encoding="utf-8") as f:
            json.dump(tag_tree_recs, f, ensure_ascii=False, indent=2)

    return tag_tree_recs


save_path = f"{working_dir}/data/{data_name}/tag_tree_recs.json"
final_tree = await build_representative_tags(tag_tree_recs, save_path)
#print(json.dumps(final_tree, ensure_ascii=False, indent=2))
print(f"Saved to : {save_path}")

# Update Graph

In [5]:
import pandas as pd      # only needed if you want to manipulate the result further
from typing import List, Tuple, Dict, Any, Optional
from generate_tag_graph2 import generate_tag, embed_tags, get_tag_group, generate_representative_tag, _embed_pool_context, generate_tag_tree
import os, json, random, re
import torch
from hierarchical_kmeans import HierarchicalKMeans

"""
Minimal debugging run. Adjust model names to ones you have locally.
- For vLLM (generation): use a small instruct model, e.g. "Qwen2.5-3B-Instruct" or "meta-llama/Meta-Llama-3-8B-Instruct".
- For embeddings: the user’s reference model "intfloat/e5-mistral-7b-instruct" works with SentenceTransformer.
"""
# ----- CONFIG -----
working_dir = "/workspace/RMS_exp"
data_name = "smollm-corpus"
#data_name = "test"
GEN_MODEL = os.environ.get("VLLM_MODEL", "/workspace/qwen7b")
EMB_MODEL = os.environ.get("EMB_MODEL", "intfloat/e5-mistral-7b-instruct")
SAVE_EMB = f"{working_dir}/data/{data_name}/key_embeddings.pt"
N_GROUP = 5000
N_TAG_SAMPLE = 6
tensor_parallel_size = 1
num_instances = 1
device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)
    
random.seed(0)

'''
# ----- SAMPLE KEYS (replace with yours) -----
sample_keys = [
    "Graph-based retrieval augmentation for enterprise documents",
    "Reward models for search result re-ranking",
    "Neural sparse indexing for web-scale retrieval",
    "Semantic tagging of legal contracts",
    "LLM orchestration with multi-agent planners",
    "Efficient RAG with vector + keyword hybrid",
    "Biomedical literature triage with MeSH terms",
    "GPU-efficient vLLM serving on multi-GPU",
    "Evaluation of negotiation agents with self-play",
    "Knowledge graph construction from PDFs",
]
'''

csv_name = "df_small.csv"
#ds = load_from_disk(f"./data/{save_name}") 
#df = ds.to_pandas()
df = pd.read_csv(f"./data/{data_name}/{csv_name}")
keys = df['text'].to_list()

INFO 10-06 05:18:48 [__init__.py:241] Automatically detected platform cuda.


In [6]:
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

try:
    from pynndescent import NNDescent  # type: ignore
except ImportError:
    NNDescent = None

try:
    import hnswlib  # type: ignore
except ImportError:
    hnswlib = None

try:
    import igraph as ig  # type: ignore
    import leidenalg as la  # type: ignore
except ImportError:
    ig = None
    la = None


K_MUTUAL = 16
K_HNSW = 8
SPILL_SIM_THRESHOLD = 0.82
MAX_SPILL_TAG_IDS = 8


def _load_tag_tree(default_path: Path) -> List[Dict[str, Any]]:
    if "tag_tree_recs" in globals() and isinstance(globals()["tag_tree_recs"], list):
        return globals()["tag_tree_recs"]
    with default_path.open() as f:
        return json.load(f)


def _flatten_tree(tree: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Tuple[int, int]]]:
    nodes: List[Dict[str, Any]] = []
    tree_edges: List[Tuple[int, int]] = []

    def dfs(node: Dict[str, Any], path: List[int], parent_id: Optional[int]) -> None:
        node_id = len(nodes)
        record = {
            "node_id": node_id,
            "path": path[:],
            "tag": node.get("tag") or "",
            "tag_ids": [int(x) for x in node.get("tag_ids", [])],
            "tags": node.get("tags", []),
            "parent_id": parent_id,
            "child_ids": [],
        }
        nodes.append(record)
        if parent_id is not None:
            tree_edges.append((parent_id, node_id))
            nodes[parent_id]["child_ids"].append(node_id)
        for idx, child in enumerate(node.get("children", [])):
            dfs(child, path + [idx], node_id)

    for idx, root in enumerate(tree):
        dfs(root, [idx], None)

    for record in nodes:
        record["depth"] = max(0, len(record["path"]) - 1)
        record["is_leaf"] = len(record["child_ids"]) == 0

    return nodes, tree_edges


def _node_text(record: Dict[str, Any]) -> str:
    tag = record.get("tag") or "general"
    tags = record.get("tags") or []
    if tags:
        sample = ", ".join(tags[:5])
        return f"{tag} ::: {sample}"
    return tag


def _embed_nodes(nodes: List[Dict[str, Any]], model_name: str, cache_path: Path, batch_size: int = 64) -> np.ndarray:
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        if isinstance(cached, dict) and "embeddings" in cached:
            cached = cached["embeddings"]
        if isinstance(cached, torch.Tensor):
            cached_np = cached.cpu().numpy()
        else:
            cached_np = np.asarray(cached)
        if cached_np.shape[0] == len(nodes):
            return cached_np.astype(np.float32)

    if SentenceTransformer is None:
        raise RuntimeError("sentence-transformers is required to embed tag graph nodes.")

    texts = [_node_text(node) for node in nodes]
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=len(texts) > batch_size,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(torch.from_numpy(embeddings), cache_path)
    return embeddings


def _normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return embeddings / norms


def _build_mutual_knn(embeddings: np.ndarray, k: int) -> Tuple[Dict[Tuple[int, int], float], Dict[Tuple[int, int], float], List[set]]:
    n = embeddings.shape[0]
    if n <= 1 or k <= 0:
        return {}, {}, [set() for _ in range(n)]

    k = min(k, n - 1)
    if NNDescent is not None:
        index = NNDescent(embeddings, n_neighbors=k + 1, metric="cosine", random_state=42)
        knn_ind, knn_dist = index.neighbor_graph
        knn_ind = knn_ind[:, 1:]
        knn_dist = knn_dist[:, 1:]
    else:
        sims = embeddings @ embeddings.T
        np.fill_diagonal(sims, -1.0)
        knn_ind = np.argsort(-sims, axis=1)[:, :k]
        # convert cos distance into similarity
        knn_dist = 1.0 - sims[np.arange(n)[:, None], knn_ind]

    neighbor_sets = [set(map(int, row.tolist())) for row in knn_ind]
    mutual: Dict[Tuple[int, int], float] = {}
    for i in range(n):
        for idx, j in enumerate(knn_ind[i]):
            j = int(j)
            if i == j:
                continue
            if i in neighbor_sets[j]:
                pair = (min(i, j), max(i, j))
                sim = 1.0 - float(knn_dist[i][idx])
                prev = mutual.get(pair)
                if prev is None or sim > prev:
                    mutual[pair] = sim

    snn: Dict[Tuple[int, int], float] = {}
    for i in range(n):
        for j in neighbor_sets[i]:
            if i >= j:
                continue
            overlap = float(len(neighbor_sets[i] & neighbor_sets[j]))
            if overlap > 0:
                snn[(i, j)] = overlap / float(k)

    return mutual, snn, neighbor_sets


def _build_hnsw_edges(embeddings: np.ndarray, k: int) -> Dict[Tuple[int, int], float]:
    if hnswlib is None:
        return {}
    n, dim = embeddings.shape
    if n <= 1 or k <= 0:
        return {}
    k = min(k, n - 1)
    index = hnswlib.Index(space="cosine", dim=dim)
    index.init_index(max_elements=n, ef_construction=200, M=32)
    index.add_items(embeddings, np.arange(n))
    index.set_ef(min(64, n - 1))
    labels, dists = index.knn_query(embeddings, k=k + 1)
    edges: Dict[Tuple[int, int], float] = {}
    for i in range(n):
        for lbl, dist in zip(labels[i], dists[i]):
            j = int(lbl)
            if i == j:
                continue
            pair = (min(i, j), max(i, j))
            sim = 1.0 - float(dist)
            prev = edges.get(pair)
            if prev is None or sim > prev:
                edges[pair] = sim
    return edges


def _build_spill_edges(nodes: List[Dict[str, Any]], embeddings: np.ndarray) -> Tuple[Dict[Tuple[int, int], float], Dict[int, set]]:
    spill_edges: Dict[Tuple[int, int], float] = {}
    spill_assignments: Dict[int, set] = {node["node_id"]: set() for node in nodes}
    for node in nodes:
        child_ids = node["child_ids"]
        if len(child_ids) < 2:
            continue
        child_vecs = embeddings[child_ids]
        sims = child_vecs @ child_vecs.T
        for idx_a, child_a in enumerate(child_ids):
            for idx_b in range(idx_a + 1, len(child_ids)):
                child_b = child_ids[idx_b]
                sim = float(sims[idx_a, idx_b])
                if sim < SPILL_SIM_THRESHOLD:
                    continue
                pair = (min(child_a, child_b), max(child_a, child_b))
                prev = spill_edges.get(pair)
                if prev is None or sim > prev:
                    spill_edges[pair] = sim
                node_a = nodes[child_a]
                node_b = nodes[child_b]
                if node_a["is_leaf"] and node_b.get("tag_ids"):
                    spill_assignments[child_a].update(node_b["tag_ids"][:MAX_SPILL_TAG_IDS])
                if node_b["is_leaf"] and node_a.get("tag_ids"):
                    spill_assignments[child_b].update(node_a["tag_ids"][:MAX_SPILL_TAG_IDS])
    return spill_edges, spill_assignments


def _aggregate_counts(nodes: List[Dict[str, Any]]) -> None:
    totals = [len(node.get("tag_ids", [])) for node in nodes]
    for node in reversed(nodes):
        node_id = node["node_id"]
        total = totals[node_id]
        for child in node["child_ids"]:
            total += totals[child]
        totals[node_id] = total
    for node in nodes:
        node["num_tag_ids"] = totals[node["node_id"]]


def _detect_communities(num_nodes: int, weighted_edges: Dict[Tuple[int, int], float]) -> List[int]:
    if num_nodes == 0:
        return []
    if ig is not None and la is not None and weighted_edges:
        graph = ig.Graph(n=num_nodes)
        graph.add_edges(list(weighted_edges.keys()))
        graph.es["weight"] = list(weighted_edges.values())
        part = la.find_partition(
            graph,
            la.RBConfigurationVertexPartition,
            weights="weight",
            resolution_parameter=1.0,
        )
        return list(part.membership)

    parent = list(range(num_nodes))
    rank = [0] * num_nodes

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    if weighted_edges:
        weights = list(weighted_edges.values())
        threshold = float(np.median(weights))
        for (u, v), w in weighted_edges.items():
            if w >= threshold:
                union(u, v)

    next_id = 0
    label_map: Dict[int, int] = {}
    membership: List[int] = []
    for idx in range(num_nodes):
        root = find(idx)
        if root not in label_map:
            label_map[root] = next_id
            next_id += 1
        membership.append(label_map[root])
    return membership


def build_tag_graph(
    working_dir: str,
    data_name: str,
    embed_model: str,
) -> Dict[str, Any]:
    base_dir = Path(working_dir) / "data" / data_name
    tag_tree_path = base_dir / "tag_tree_recs.json"
    tag_tree = _load_tag_tree(tag_tree_path)

    nodes, tree_edges = _flatten_tree(tag_tree)
    _aggregate_counts(nodes)

    cache_path = base_dir / "tag_graph_node_embeddings.pt"
    embeddings = _embed_nodes(nodes, embed_model, cache_path)
    embeddings = _normalize_embeddings(embeddings)

    mutual, snn, _ = _build_mutual_knn(embeddings, K_MUTUAL)
    hnsw = _build_hnsw_edges(embeddings, K_HNSW)
    spill_edges, spill_assignments = _build_spill_edges(nodes, embeddings)

    for node_id, extras in spill_assignments.items():
        if not extras:
            continue
        original = set(nodes[node_id].get("tag_ids", []))
        spill_only = sorted(int(x) for x in extras if x not in original)
        if spill_only:
            nodes[node_id]["spill_tag_ids"] = spill_only

    edge_map: Dict[Tuple[int, int], Dict[str, Any]] = {}

    def add_edge(u: int, v: int, kind: str, weight: float) -> None:
        if u == v:
            return
        key = (min(u, v), max(u, v))
        entry = edge_map.setdefault(key, {"source": key[0], "target": key[1], "kinds": {}})
        entry["kinds"][kind] = float(weight)

    for parent, child in tree_edges:
        add_edge(parent, child, "tree", 1.0)

    for (u, v), weight in mutual.items():
        add_edge(u, v, "mutual_knn", weight)

    for (u, v), weight in snn.items():
        add_edge(u, v, "snn", weight)

    for (u, v), weight in hnsw.items():
        add_edge(u, v, "hnsw", weight)

    for (u, v), weight in spill_edges.items():
        add_edge(u, v, "spill", weight)

    weighted_edges: Dict[Tuple[int, int], float] = {}
    for key, entry in edge_map.items():
        weight = 0.0
        for kind, value in entry["kinds"].items():
            if kind == "tree":
                weight += 1.0
            elif kind == "mutual_knn":
                weight += value
            elif kind == "snn":
                weight += value
            elif kind == "hnsw":
                weight += 0.5 * value
            elif kind == "spill":
                weight += value
        weighted_edges[key] = weight

    communities = _detect_communities(len(nodes), weighted_edges)

    nodes_out: List[Dict[str, Any]] = []
    for node in nodes:
        record = {
            "node_id": node["node_id"],
            "path": node["path"],
            "tag": node.get("tag", ""),
            "depth": node["depth"],
            "parent_id": node["parent_id"],
            "child_ids": node["child_ids"],
            "tag_ids": node.get("tag_ids", []),
            "spill_tag_ids": node.get("spill_tag_ids", []),
            "tags": node.get("tags", []),
            "num_tag_ids": node.get("num_tag_ids", 0),
            "is_leaf": node["is_leaf"],
            "community": communities[node["node_id"]] if communities else 0,
        }
        nodes_out.append(record)

    edges_out = []
    kind_counts: Dict[str, int] = {}
    for entry in edge_map.values():
        edges_out.append({
            "source": entry["source"],
            "target": entry["target"],
            "kinds": entry["kinds"],
        })
        for kind in entry["kinds"].keys():
            kind_counts[kind] = kind_counts.get(kind, 0) + 1

    tag_graph = {
        "nodes": nodes_out,
        "edges": edges_out,
        "meta": {
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "working_dir": working_dir,
            "data_name": data_name,
            "embedding_model": embed_model,
            "node_count": len(nodes_out),
            "edge_count": len(edges_out),
            "edge_kind_counts": kind_counts,
            "params": {
                "k_mutual": K_MUTUAL,
                "k_hnsw": K_HNSW,
                "spill_threshold": SPILL_SIM_THRESHOLD,
                "max_spill_tag_ids": MAX_SPILL_TAG_IDS,
            },
            "dependencies": {
                "pynndescent": bool(NNDescent),
                "hnswlib": bool(hnswlib),
                "leidenalg": bool(la),
            },
            "embeddings_path": str(cache_path),
        },
    }
    return tag_graph


base_dir = Path(working_dir) / "data" / data_name
base_dir.mkdir(parents=True, exist_ok=True)
tag_graph_path = base_dir / "tag_graph.json"

if not base_dir.exists():
    raise FileNotFoundError(f"Data directory not found: {base_dir}")

tag_graph = build_tag_graph(working_dir, data_name, EMB_MODEL)

with tag_graph_path.open("w", encoding="utf-8") as f:
    json.dump(tag_graph, f, ensure_ascii=False, indent=2)

print(
    f"Saved tag_graph with {len(tag_graph['nodes'])} nodes and {len(tag_graph['edges'])} edges -> {tag_graph_path}"
)


config_sentence_transformers.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Saved tag_graph with 5070 nodes and 40587 edges -> /workspace/RMS_exp/data/smollm-corpus/tag_graph.json


## Test

In [ ]:
# Generate Representative Tag

def make_leaf_tag_recs(tag_tree_recs, tag_recs):

    def walk(node):
        for tag_dict in node:
            if tag_dict["tag_ids"] != []:
                tags = []
                for tag_id in tag_dict["tag_ids"]:
                    tag = tag_recs[tag_id]["tag"]
                    tags.append(tag)
                tag_dict["tags"] = tags
            
            if "children" in tag_dict:
                walk(tag_dict["children"])

    walk(tag_tree_recs)
    return tag_tree_recs

def get_representative_tag_request(tag_tree_recs):

    all_requests = AllRequests(max_request=50)

    def set_reqresentative_tag(tag_tree_recs, tag_ids, representative_tag):

        if tag_ids==None or tag_ids==[]:
            raise Exception("tag_ids should include at least one index inside the list")
        
        tag_tree_recs_ = tag_tree_recs
        for i, tag_id in enumerate(tag_ids):
            tag_dict = tag_tree_recs_[tag_id]

            if "children" in tag_dict:
                tag_tree_recs_ = tag_dict["children"]
            else:
                raise Exception("tag_dict doesn't have children")
            
        tag_tree_recs_[tag_ids[-1]]["tag"] = representative_tag
        
        return tag_tree_recs


    def walk(child_recs, tag_ids):
        
        generate_tag = True
        tags = []
        for id, tag_dict in enumerate(child_recs):
            if "tag" in tag_dict:
                tags.append(tag_dict["tag"])
            else:
                generate_tag = False
                walk(tag_dict["children"], tag_ids + [id])

        if generate_tag:
            if tag_ids==[]:
                return True
            else:
                sample = ["general"] if not tags else (tags if len(tags) <= n_tag_sample else random.sample(tags, n_tag_sample))
                lines = "\n".join(f"- {tag}" for tag in sample)

                prompt = f"""You are a taxonomy expert. Given the following sample tags from one cluster,
produce ONE concise representative tag (≤ 6 words) that best describes them all.\n
Do NOT include punctuation at the end. Output ONLY the tag text, nothing else.\n\n
Sample tags:\n{lines}\n\nRepresentative tag:"""
                
                request = {"tag_ids":tag_ids, "prompt":prompt}
                all_requests.add(request)


    while_end = walk(tag_tree_recs, [])

    if while_end:
        return tag_tree_recs, True

    results = await all_requests.process(max_tokens = 3000, temperature=0, save_dir = "progress_questions", restart = True)

    for result in results:
        tag_ids = result["tag_ids"]
        output = result["output"]

        tag_tree_recs = set_reqresentative_tag(tag_tree_recs, tag_ids, output)

    return tag_tree_recs, False


while_end = False

while not while_end:

    tag_tree_recs, while_end = get_representative_tag_request(tag_tree_recs)

        
with open(f"data/{data_dir}/tag_tree_recs.json", "w") as f:
    json.dump(tag_tree_recs, f)  # {request_id: {"titles":[...], "keywords":["keyword1", ...], "questions":["question1", ...], "irr_questions":["irr_question1", ...]}}



# Make Dataset (Make questions -> Get top relevant keys -> Judge by LLM)

## Async Engine Class

In [ ]:
!pip install vllm==0.6.5
import os
os.environ["OMP_NUM_THREADS"] = "4"

In [ ]:
# Use Model from local environment

import asyncio
from transformers import AutoTokenizer
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams
import time, warnings
import os
from tqdm import tqdm

model_name = "/workspace/qwen7b"
tensor_parallel_size = 2
pipeline_parallel_size = 1
data_parallel_size = 2

engine_args = AsyncEngineArgs(
    model = model_name,
    tensor_parallel_size = tensor_parallel_size,
    pipeline_parallel_size = pipeline_parallel_size,
    data_parallel_size = data_parallel_size,
    gpu_memory_utilization=0.95,
    disable_log_stats=True,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)
engine.log_requests = False
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

from jsonschema import validate, ValidationError

def validate_item(item, schema):
    try:
        validate(item, schema)
        return True, None
    except ValidationError as e:
        return False, str(e)


class AllRequests:
    
    def __init__(self, max_request):
        self.max_request = max_request
        self.requests = []
        self.request_ids = []
        self.request_id = 0
        self.results = []
        self.finished_ids = []
        self.progress_bar = None
        
    def add(self, request):
        self.requests.append(request)
        self.request_ids.append(self.request_id)
        self.request_id += 1
    
    async def process(self, model=model_name, max_tokens = 3000, temperature=0.4, save_dir = "progress_log", restart = False):

        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        if restart:
            if os.path.exists(f"{save_dir}/finished_ids.json") and os.path.exists(f"{save_dir}/results.json"):
                with open(f"{save_dir}/finished_ids.json") as f:
                    finished_ids = json.load(f)
                with open(f"{save_dir}/results.json") as f:
                    self.results = json.load(f)
                for finished_id in finished_ids:
                    self.finished_ids.append(finished_id)
                    id = self.request_ids.index(finished_id)
                    self.request_ids.pop(id)
                    self.requests.pop(id)
        else:
            if os.path.exists(f"{save_dir}/finished_ids.json"):
                os.remove(f"{save_dir}/finished_ids.json")
            if os.path.exists(f"{save_dir}/results.json"):
                os.remove(f"{save_dir}/results.json")

        total = len(self.requests)
        self.progress_bar = tqdm(total=total, desc="Processing Requests")

        await asyncio.gather(
            *[self.process_requests(temperature = temperature, max_tokens = max_tokens, save_dir=save_dir) for _ in range(self.max_request)]
        )

        self.progress_bar.close()
            
        return self.results


    async def process_requests(self, max_tokens = 3000, temperature=0.4, save_dir = "progress_log"):

        while len(self.requests) != 0:
            request_dict = self.requests.pop(0)
            request_id = self.request_ids.pop(0)

            prompt = request_dict["prompt"]

            final_output = None
            results_generator = engine.generate(prompt, SamplingParams(temperature=temperature, max_tokens=max_tokens), request_id)
            async for request_output in results_generator:
                # print(request_output) => for streaming
                final_output = request_output

            output = final_output.outputs[0].text
            
            request_dict["output"] = output

            valid = False
            try:
                json.loads(json.dumps(request_dict))
                valid = True
            except Exception as e:
                print("\nrequest id : ", request_id)
                print(e)
            #valid, error = validate_item(request_dict, schema)

            if valid:
                self.results.append(request_dict)
                self.finished_ids.append(request_id)
    
                with open(f"{save_dir}/results.json", "w") as f:
                    json.dump(self.results, f)
                with open(f"{save_dir}/finished_ids.json", "w") as f:
                    json.dump(self.finished_ids, f)
    
            self.progress_bar.update(1)
            

## Make queries

In [ ]:
# Make questions

prompts = []
all_requests = AllRequests(max_request=50)

import os, re, json
import pandas as pd      # only needed if you want to manipulate the result further
from datasets import load_from_disk

#save_name = "ja-web-text-2"
save_name = "smollm-corpus"
csv_name = "df_small.csv"
#ds = load_from_disk(f"./data/{save_name}") 
#df = ds.to_pandas()
df = pd.read_csv(f"./data/{save_name}/{csv_name}")
df

In [ ]:
for request_id in range(len(df)):

    system = f"""You are a helpful assistant who extracts titles, and keywords from a sentence provided by the user, and also creates questions and irrelevant questions.
Following the user's instructions, analyze the content of the sentence and respond according to the output format below.
Make sure that your questions are creative and sometimes that asks question

Output format:
<titles>["Title1", "Title2", ... ]</titles>
<keywords>["Keyword1", "Keyword2", ... ]</keywords>
<questions>["Question1", "Question2", ... ]</questions>
<irrelevant questions>["Question1", "Question2", ... ]</irrelevant questions>"""

    user_prompt = f"""Sentence:
'''
{df.iloc[request_id]["text"]}
'''

Instructions:
1. Summarize the content of the sentence into 2-3 one-line titles.
2. Extract 3–5 main keywords from the sentence.
3. Create several questions and irrelevant ones about the sentence, ranging from easy to difficult.
4. Enclose each element in order with the tags <titles></titles>, <keywords></keywords>, <questions></questions>, and <irrelevant questions></irrelevant questions> when outputting.

Follow the instructions step-by-step and think in sequence."""


    prompt = tokenizer.apply_chat_template(
                [{"role": "system", "content":system}, {"role": "user", "content":user_prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
    
    request = {"request_id":request_id, "prompt":prompt}
    all_requests.add(request)

results = await all_requests.process(max_tokens = 3000, temperature=0, save_dir = "progress_questions", restart = True)
    
    

In [ ]:
# process result
import re, json

save_dir = "progress_questions"
data_dir = "smollm-corpus"

with open(f"{save_dir}/results.json") as f:
    results = json.load(f)
    
def extract_text(textC: str, textB: str) -> str | None:
    # Build a non-greedy pattern like r'<tag>(.*?)</tag>'
    pattern = rf"<{re.escape(textB)}>(.*?)</{re.escape(textB)}>"
    match = re.search(pattern, textC, flags=re.DOTALL)
    return match.group(1) if match else None

def extract_int(text: str) -> int | None:
    """
    Return the first integer found in `text`.
    If no digits appear, return None.
    """
    match = re.search(r"-?\d+", text)  # handles negative numbers too
    return int(match.group()) if match else None

query_dict = {}  # {request_id: {"title":"", "summary":"", "keywords":["keyword1", ...], "questions":["question1", ...]}}

for result in results:
    request_id = result["request_id"]
    output = result["output"]
    titles = extract_text(output, "titles")
    keywords = extract_text(output, "keywords")
    questions = extract_text(output, "questions")
    irr_questions = extract_text(output, "irrelevant questions")

    #print()
    #print("questions: ", questions)

    try:
        titles = json.loads(titles)
        keywords = json.loads(keywords)
        questions = json.loads(questions)
        irr_questions = json.loads(irr_questions)
        query_dict[request_id] = {"titles":titles, "keywords":keywords, "questions":questions, "irr_questions":irr_questions}
    except:
        query_dict[request_id] = {"titles":[], "keywords":[], "questions":[], "irr_questions":[]}
        continue
    
with open(f"data/{data_dir}/query_dict.json", "w") as f:
    json.dump(query_dict, f)  # {request_id: {"titles":[...], "keywords":["keyword1", ...], "questions":["question1", ...], "irr_questions":["irr_question1", ...]}}



### Other prompts

In [ ]:
    # English version
    system = f"""You are a helpful assistant who makes questions about a sentence given by a user.
Please follow the instructions the user will give you, analyze the content of the sentence, and answer your output following the output format.

Output format:
<questions>["question 1", "question 2", ... ]</questions>"""

    user_prompt = f"""Sentence:
'''
{df.iloc[request_id]["normalized_text"]}
'''

Instructions:
1. Think of some questions about the sentence. Note that the questions range from easy to difficult ones.
2. Please return your output enclosing your questions within <questions></questions> tag.

Let's think step by step following each step of the instructions."""


    # Japanese version
    system = f"""あなたは、ユーザーから与えられた文について質問を作成する親切なアシスタントです。
ユーザーが与える指示に従い、文の内容を分析し、出力形式に従って答えてください。

出力形式:
<questions>["質問1", "質問2", ... ]</questions>"""

    user_prompt = f"""文:
'''
{df.iloc[request_id]["normalized_text"]}
'''

指示:
1. 文についていくつかの質問を考えてください。質問は簡単なものから難しいものまで含めてください。
2. 質問を<questions></questions>タグで囲んで出力してください。

ステップごとに指示に従って順を追って考えましょう。"""


    # Japanese version 2
    system = f"""あなたは、ユーザーから与えられた文について
1) タイトルを考え、
2) 要約を作り、
3) 質問を作成する親切なアシスタントです。
ユーザーが与える指示に従い、文の内容を分析し、
下記の出力形式に従って答えてください。

出力形式:
<titles>["タイトル1", "タイトル2", ... ]</titles>
<summaries>["要約1", "要約2", ... ]</summaries>
<questions>["質問1", "質問2", ... ]</questions>"""

    user_prompt = f"""文:
'''
{df.iloc[request_id]["normalized_text"]}
'''

指示:
1. 文の主題がわかる簡潔なタイトルを 1〜3 個考えてください。
2. 文全体の要旨を 1〜2 文で要約してください（複数可）。
3. 文について簡単なものから難しいものまで、いくつか質問を作ってください。
4. 生成したタイトル・要約・質問はそれぞれ
   <titles>…</titles>・<summaries>…</summaries>・<questions>…</questions>
   のタグで必ず囲んで出力してください。

ステップごとに指示に従い、順を追って考えましょう。"""

## Reward Model Gets TopN-Relevant Sentences and LLM Judges Which Content Is More Relevant to A Question -> Got some error

In [ ]:
!pip install -U vllm==0.10.1

In [ ]:
# Get top N files

from itertools import combinations
import random, json
import os, re
import torch
from copy import deepcopy
from rmsearch import Search
import os, re, json
import pandas as pd      # only needed if you want to manipulate the result further
from datasets import load_from_disk
from vllm_reward2 import build_llm, search
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # do this ONCE per kernel before using mp
os.environ.setdefault("VLLM_CONFIGURE_LOGGING", "1")  # don't let vLLM set up its own handlers

save_name = "smollm-corpus"
log_save_dir = "progress_search_log"
model_name = f"/workspace/llama3b-rm-converted-model"
df = pd.read_csv(f"./data/{save_name}/df_small.csv")
with open(f"./data/{save_name}/query_dict.json") as f:
    query_dict = json.load(f)
#with open("tag_dict.json") as f:
#    tag_dict = json.load(f)  # [{"tag":"", "children":[{}, ...]}, ...]
with open(f"./data/{save_name}/tag_tree_recs.json") as f:
    tag_dict = json.load(f)  # [{"tag":"", "children":[{}, ...]}, ...]

# GPUs = tensor_parallel_size * num_instances
tensor_parallel_size = 1  # Assign Model Parameters Over GPUS
num_instances = 1         # Number of Models

# device_groups is set according to gpu environment
# e.g. device_groups = [[0], [1]]  # 2 workers on GPU 0 and 1
device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)

rm = build_llm(
    model_name=model_name,
    tensor_parallel_size=len(device_groups[0]),
    num_instances=len(device_groups),
    device_groups=device_groups,
    max_model_len=4000,
    max_num_seqs=64,
    gpu_memory_utilization=0.90,
    runner="pooling",
)

tokenizer = rm.tokenizer

'''
# ── Search engine instance ────────────────────────────────────────────────────
search_engine = Search(
    model_name="/workspace/llama3b-rm-converted-model",
    tensor_parallel_size=1,
    pipeline_parallel_size=4,
    max_request = 10000,
)

tokenizer = search_engine.tokenizer
'''


In [ ]:
#df = pd.read_csv(f"./df_all.csv")
#n_sample = 4672
#n_sample = 2000
#df = df[:n_sample]
#search_engine.max_request = None

"""
def llm_template_func(query, key):
    message = [
        {'role': 'user', 'content':f"Generate tag for the sentence\n\nSentence:'''{query}'''"},
        {'role': 'assistant', 'content':f"{key}"}
    ]
    if len(message[0]["content"]) > 4000: message[0]["content"] = message[0]["content"][:4000]+"..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    #prompt = prompt[17:]
    return prompt
"""

from tqdm import tqdm

def llm_template_func(row):
    query, tag = row["query"], row["key"]
    
    message = [
        {'role': 'user', 'content':f"Generate tag for the sentence\n\nSentence:'''{query}'''"},
        {'role': 'assistant', 'content':f"{tag}"}
    ]
    if len(message[0]["content"]) > 4000: message[0]["content"] = message[0]["content"][:4000]+"..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    #prompt = prompt[17:]
    return prompt

'''
async def assign_tag(tag_dict, key_dict): # -> key_dict : [{"key":"", "tag_ids":[0,3, ...], **kwargs}]
    # tag_dict: [{"tag":"", "children":[{}, ...]}, ...] 
    # key_dict: [{"key":"", **kwargs}, ...]
    # llm_template: function (key_dict -> str)

    queries = []
    for dict in key_dict:
        key = dict["key"]
        queries.append(key) # when assigning tag to each key, rmsearch gets tags from keys. Here key plays as queries in rmsearch.
    
    tag_ids_list = [[] for i in range(len(key_dict))]
    end = False
    while True:
        tags = []
        for tag_ids in tag_ids_list:
            tag = get_tag(tag_ids, tag_dict)
            if not tag:
                end = True
                break
            tags.append(tag)
        
        if end: break

        output = await search_engine(queries, tags, k=1, return_relevance=True)  # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]

        for i, output_dict in enumerate(output):
            new_tag_id = output_dict["keys"][0]["key_id"]
            tag_ids_list[i].append(new_tag_id)

    tag2key = deepcopy(tag_dict)  # [{"tag":"", "key_ids":[], "children":[{"key_id":[]}]}]
    for i in range(len(key_dict)):
        set_key_id(tag2key, tag_ids_list[i], i)
        key_dict[i]["tag_ids"] = tag_ids_list[i]

    return key_dict, tag2key
'''


async def search_tag(query_dict, tag_dict, k_tag=2):
    # Args
    # query_dict: [{"query":"", **kwargs}, ...]
    # tag_dict: [{"tag":"", "children":[{}, ...]}, ...]

    # Return
    # query2tag_ids : [{"tag_ids":[[0,3,...], [2,1,...]]}, ...]   # (num_queries, "tag_ids":(k_tag, depth))
    # tag2query : [{"tag":"", "key_ids":[0,2, ...], "children":[{"key_ids":[2, ...]}, {"key_ids":[0, ...]}]}]
    
    # Output definition
    tag2query = deepcopy(tag_dict)
    query2tag_ids = [{"tag_ids":[[] for _ in range(k_tag)]} for i in range(len(query_dict))]   # query2tag_ids = [{"tag_ids":[[]]}, ...]   # (num_queries, "tag_ids":(k_tag, depth))

    # Parameters definition
    tags = [tag2query_dict["tag"] for tag2query_dict in tag_dict]
    tags_request = [{"tags":[tags]} for _ in range(len(query_dict))]    # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))
    while_end = False
    depth = 1

    def get_tag_dict(tag_ids, tag_dict):
        if tag_ids == []: return None
    
        for tag_id in tag_ids[:-1]:
            if "children" not in tag_dict[tag_id]:
                return None
            else:
                tag_dict = tag_dict[tag_id]["children"]
        
        return tag_dict[tag_ids[-1]]
    
    def get_tag(tag_ids, tag_dict):
        
        if tag_ids == []: return None
    
        for tag_id in tag_ids[:-1]:
            tag_dict = tag_dict[tag_id]["children"]
        
        return tag_dict[tag_ids[-1]]["tag"]
    
    def set_key_id(tag2key, tag_ids, key_id):
        for i, tag_id in enumerate(tag_ids):
            tag_dict = tag2key[tag_id]
            if "key_ids" in tag_dict:
                tag_dict["key_ids"].append(key_id)
            else:
                tag_dict["key_ids"] = [key_id]
    
            if len(tag_ids) > i + 1:
                tag2key = tag_dict["children"]
        
        return tag2key
    
    def set_query_id(tag2query, tag_ids, query_id):
        tag2query_ = tag2query
        for i, tag_id in enumerate(tag_ids):
            tag_dict = tag2query_[tag_id]
            if "query_ids" in tag_dict:
                tag_dict["query_ids"].append(query_id)
            else:
                tag_dict["query_ids"] = [query_id]
    
            #if len(tag_ids) > i + 1:
            #    tag2query_ = tag_dict["children"]

            if "children" in tag_dict:
                tag2query_ = tag_dict["children"]
        
        return tag2query

    

    # Main loop
    while not while_end:

        requests = [] # [{"group": query_id, "query":, "tag":tag, "tag_id":tag_id, "k":k_tag, "nth_tag":nth_tag}, ...]
        query_and_n_top_ids = [] # [(query_id, n_top), ...] # (num_request)
        total_requests = 0

        for query_id in range(len(tags_request)):
            for nth_tag, tag_list in enumerate(tags_request[query_id]["tags"]):
                query_and_n_top_ids.append((query_id, nth_tag))
                #requests += [{"group": query_id, "query":query_dict[query_id]["query"], "tag":tag, "tag_id":tag_id, "k":k_tag, "nth_tag":nth_tag} for tag_id, tag in enumerate(tag_list)]
                requests.append({"query":query_dict[query_id]["query"], "keys":tag_list, "k": k_tag, "return_relevance":True})
                total_requests += len(tag_list)

        
        key_str = False
        if os.path.exists(f"output{depth}.json"):
            with open(f"output{depth}.json") as f:
                output = json.load(f)
            key_str = True
        else:
            batch_size = total_requests // num_instances
            print(f"Graph Depth: {depth},  total_requests: {total_requests},  Batch size: {batch_size}")
            output = search(rm, requests, llm_template_func, topk = k_tag, batch_size=1000, timeout_s=10000)

            with open(f"output{depth}.json", "w") as f:
                json.dump(output, f)
        '''
        
        batch_size = total_requests // num_instances
        print(f"Graph Depth: {depth},  total_requests: {total_requests},  Batch size: {batch_size}")
        output = search(rm, requests, llm_template_func, topk = k_tag, batch_size=1000, timeout_s=10000)

        with open(f"output{depth}.json", "w") as f:
            json.dump(output, f)
        '''
        
        # output : [{"query":, "query_id":, "keys": [{"key":, "key_id":, "relevance":}, ....]}, ...] 

        # print(df.head())
        # Defalt:              output : [{"query":, "tag_id": ,"tag":, "k":, "relevance":}, ...] # (k)
        # if topk_with_group:  output : {group: [{"query":, "tag_id": ,"tag":, "k":, "relevance":}, ...], ...} # (n_group, k)
        #search_engine.progress_bar = tqdm(total=len(df), desc="Processing Requests")
        #output = await search_engine.search_by_df(df, topk_with_group=True, progress_save_dir = log_save_dir)

        tags_request = [{"tags":[]} for _ in range(len(query_dict))] # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))

        result1 = {query_id:{"tag_ids_list":[], "relevance_list":[]} for query_id in range(len(query_dict))}   # {"query_id":{"tag_ids_list":[[3,1],],"relevance_list":[]}}
        for reuqest_id, output_dict in enumerate(output):
            query_id, nth_tag_ids = query_and_n_top_ids[reuqest_id]
            tags = []
            tag_relevance = []
            pre_tag_ids = query2tag_ids[query_id]["tag_ids"][nth_tag_ids]

            #print()
            #print("request ", reuqest_id)
            #print(query_id, nth_tag_ids)
            
            for top_nth in range(k_tag):
                try:  # output[query_id]["keys"][top_nth] can be index out of range. It goes to next output_dict.
                    new_tag_id = output_dict["keys"][top_nth]["key_id"]
                    relevance = output_dict["keys"][top_nth]["relevance"]
                    #new_tag_id = output[query_id]["keys"][top_nth]["key_id"]
                    #relevance = output[query_id]["keys"][top_nth]["relevance"]
                    #print("success")
                    #print(pre_tag_ids, new_tag_id)
                    result1[query_id]["tag_ids_list"].append(pre_tag_ids+[new_tag_id])
                    result1[query_id]["relevance_list"].append(relevance)
                except Exception as e:
                    #print("error")
                    continue

        while_end = True

        for query_id in result1:
            tag_relevance = result1[int(query_id)]["relevance_list"]
            tag_ids_list = result1[int(query_id)]["tag_ids_list"]
            #print("query_id ", query_id)
            #print("tag_relevance: ", tag_relevance)
            #print("tag_ids_list: ", tag_ids_list)
            if len(tag_relevance) == 0:
                top_tag_ids_list = []
            if len(tag_relevance) < k_tag:  # if there are not enough tag_ids
                _, indices = torch.topk(torch.tensor(tag_relevance), k=len(tag_relevance))
                top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]
            else:  # normal pattern
                _, indices = torch.topk(torch.tensor(tag_relevance), k=k_tag)
                top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]
                
            #query2tag_ids[int(query_id)]["tag_ids"] = top_tag_ids_list
            #print("top_tag_ids_list: ", top_tag_ids_list)

            new_tag_ids_list = []
            for tag_ids in top_tag_ids_list:
                tag_dict_ = get_tag_dict(tag_ids, tag2query)
                if tag_ids == [1, 9, 7]:
                    pass
                    #print(tag_ids)
                    #print(tag_dict_)
                    
                if tag_dict_==None:
                    print("get_tag_dict(tag_ids[:-1], tag2query): ", get_tag_dict(tag_ids[:-1], tag2query))
                    print("tag_ids: ", tag_ids)
                    raise Exception("tag_dict_ shouldn't be None. Something in the code might be not working.")
                elif "children" not in tag_dict_:
                    continue
                elif tag_dict_["children"]==[]:
                    continue

                #if tag_ids == [1, 9, 7]:
                #    raise Exception("error")

                tags = []
                for j in range(len(tag_dict_["children"])):
                    tag = tag_dict_["children"][j]["tag"]
                    tags.append(tag)

                while_end = False
                tags_request[int(query_id)]["tags"].append(tags)
                new_tag_ids_list.append(tag_ids)

            query2tag_ids[int(query_id)]["tag_ids"] = new_tag_ids_list

        depth += 1

        '''

        tags_request = [{"tags":[]} for _ in range(len(query_dict))] # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))
        while_end = True

        for request_id, output_dict in enumerate(output):
            query_id, nth_tag = query_and_n_top_ids[request_id]
            tags = []
            top_tag_ids_list = []
            for top_nth in range(k_tag):
                row = output_dict["keys"][top_nth]
                new_tag_id = row["key_id"]
                pre_tag_ids = query2tag_ids[int(query_id)]["tag_ids"][nth_tag]
                
                top_tag_ids_list.append(pre_tag_ids+[new_tag_id])
                print(pre_tag_ids+[new_tag_id])

            # update query2tag_ids
            query2tag_ids[int(query_id)]["tag_ids"] = top_tag_ids_list
            
            # Make new tag_request
            for tag_ids in top_tag_ids_list:
                tag_dict_ = get_tag_dict(tag_ids, tag_dict)
                #print(tag_dict_)

                # If tag_dict_ is either None or doesn't have children, no tag_request is generated in this brancch
                if not tag_dict_:
                    continue
                elif not "children" in tag_dict_:
                    continue

                tags = []
                for j in range(len(tag_dict_["children"])):
                    tag = tag_dict_["children"][j]["tag"]
                    tags.append(tag)

                while_end = False
                tags_request[int(query_id)]["tags"].append(tags)

        depth += 1
        '''

    for query_id in range(len(query_dict)):
        for nth_top, tag_ids in enumerate(query2tag_ids[query_id]["tag_ids"]):
            tag2query = set_query_id(tag2query, tag_ids, query_id)
            
    return query2tag_ids, tag2query

'''    
        for query_id in output:
            tags = []
            top_tag_ids_list = []
            for top_nth in range(k_tag):
                row = output[query_id][top_nth]
                nth_tag = row["nth_tag"]
                new_tag_id = row["tag_id"]
                pre_tag_ids = query2tag_ids[int(query_id)]["tag_ids"][nth_tag]
                
                top_tag_ids_list.append(pre_tag_ids+[new_tag_id])

            # update query2tag_ids
            query2tag_ids[int(query_id)]["tag_ids"] = top_tag_ids_list
            
            # Make new tag_request
            for tag_ids in top_tag_ids_list:
                tag_dict_ = get_tag_dict(tag_ids, tag_dict)
                if "children" not in tag_dict_:
                    continue

                tags = []
                for j in range(len(tag_dict_["children"])):
                    tag = tag_dict_["children"][j]["tag"]
                    tags.append(tag)

                while_end = False
                tags_request[int(query_id)]["tags"].append(tags)

        depth += 1

    for query_id in range(len(query_dict)):
        for nth_top, tag_ids in enumerate(query2tag_ids[query_id]["tag_ids"]):
            tag2query = set_query_id(tag2query, tag_ids, query_id)
            
    return query2tag_ids, tag2query
'''


key_dict = []
for i in range(len(df)):
    #key_dict.append({"query":df.iloc[i]["normalized_text"]})
    key_dict.append({"query":df.iloc[i]["text"]})

# key_dict: [{"query":"", **kwargs}, ...]
# tag_dict: [{"tag":"", "children":[{"tag":""}, ...]}, ...]

# search_tag function
# Args
# query_dict: [{"query":"", **kwargs}, ...]
# tag_dict: [{"tag":"", "children":[{}, ...]}, ...]
# Return
# query2tag_ids : [{"tag_ids":[[0,3,...], [2,1,...]]}, ...]   # (num_queries, "tag_ids":(k_tag, depth))
# tag2query : [{"tag":"", "key_ids":[0,2, ...], "children":[{"key_ids":[2, ...]}, {"key_ids":[0, ...]}]}]

query2tag_ids, tag2query = await search_tag(key_dict, tag_dict)

with open(f"./data/{save_name}/query2tag_ids.json", "w") as f:
    json.dump(query2tag_ids, f)

with open(f"./data/{save_name}/tag2query.json", "w") as f:
    json.dump(tag2query, f)

'''
with open("query2tag_ids.json", "w") as f:
    json.dump(query2tag_ids, f)

with open("tag2query.json", "w") as f:
    json.dump(tag2query, f)
'''

print("All Finished")


### Test

In [ ]:
save_name = "smollm-corpus"
with open(f"./data/{save_name}/query_dict.json") as f:
    query_dict = json.load(f)

print(len(query_dict))

In [ ]:
def set_query_id(tag2query, tag_ids, query_id):
    tag2query_ = tag2query
    for i, tag_id in enumerate(tag_ids):
        tag_dict = tag2query_[tag_id]
        if "query_ids" in tag_dict:
            tag_dict["query_ids"].append(query_id)
        else:
            tag_dict["query_ids"] = [query_id]

        #if len(tag_ids) > i + 1:
        #    tag2query_ = tag_dict["children"]

        if "children" in tag_dict:
            tag2query_ = tag_dict["children"]

    return tag2query

tag_dict= [{"tag":"a", "children":[{"tag":"a"}, {"tag":"b"}]}, {"tag":"b", "children":[{"tag":"a"}, {"tag":"b"}]}]
for i in range(2):
    for j in range(2):
        tag_ids = [i,j]
        tag2query = set_query_id(tag_dict, tag_ids, 2*i+j)
        
print(tag2query)

In [ ]:
def llm_template_func(query, key):
    message = [{'role': 'user', 'content':f"Give me relevance score between\n\nQuery:{query}\n\nand\n\n{key}"}]
    if len(message[0]["content"]) > 4000: message[0]["content"] = message[0]["content"][:4000]+"..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    #prompt = prompt[17:]
    return prompt

search_engine.llm_template = llm_template_func

async def search_key(queries, keys, tag2key, k_tag=2, k_key=5):
    # queries: ["...", ...]
    # keys: ["...", ...]
    # tag2key: [{"tag":"", "key_ids":[0,2, ...], "children":[{"key_ids":[2, ...]},{"key_ids":[0, ...]}, ...]}, ...]
    
    query2tag_ids = [{"tag_ids":[[] for _ in range(k_tag)]} for i in range(len(queries))]   # query2tag_ids = [{"tag_ids":[[]]}, ...]   # (num_queries, "tag_ids":(k_tag, depth))
    
    tags = [tag2key_dict["tag"] for tag2key_dict in tag2key]
    tags_request = [{"tags":[tags]} for _ in range(len(queries))]    # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))
    while_end = False

    while not while_end:

        requests = []
        query_and_n_top_ids = [] # [(query_id, n_top), ...] # (num_request)

        for query_id in range(len(tags_request)):
            for nth_tag_ids, tag_list in enumerate(tags_request[query_id]["tags"]):
                query_and_n_top_ids.append((query_id, nth_tag_ids))
                requests.append(search_engine(queries[query_id], tag_list, k=k_tag, return_relevance=True))
    
        output = await asyncio.gather(*requests) # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]
        #output = await search_engine(queries, tags, k=k_tag, return_relevance=True)  # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]

        tags_request = [{"tags":[]} for _ in range(len(queries))] # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))

        result1 = {query_id:{"tag_ids_list":[], "relevance_list":[]} for query_id in range(len(queries))}   # {"query_id":{"tag_ids_list":[[3,1],],"relevance_list":[]}}
        for reuqest_id, output_dict in enumerate(output):
            query_id, nth_tag_ids = query_and_n_top_ids[reuqest_id]
            tags = []
            tag_relevance = []
            pre_tag_ids = query2tag_ids[query_id]["tag_ids"][nth_tag_ids]

            for top_nth in range(k_tag):
                new_tag_id = output[query_id]["keys"][top_nth]["key_id"]
                relevance = output[query_id]["keys"][top_nth]["relevance"]
                result1[query_id]["tag_ids_list"].append(pre_tag_ids+[new_tag_id])
                result1[query_id]["relevance_list"].append(relevance)

        while_end = True

        for query_id in result1:
            tag_relevance = result1[query_id]["relevance_list"]
            tag_ids_list = result1[query_id]["tag_ids_list"]
            _, indices = torch.topk(torch.tensor(tag_relevance), k=k_tag)
            top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]
            query2tag_ids[query_id]["tag_ids"] = top_tag_ids_list
            for tag_ids in top_tag_ids_list:
                tag_dict = get_tag_dict(tag_ids, tag2key)
                if "children" not in tag_dict:
                    continue

                tags = []
                for j in range(len(tag_dict["children"])):
                    tag = tag_dict["children"][j]["tag"]
                    tags.append(tag)

                while_end = False
                tags_request[query_id]["tags"].append(tags)
            
    query2key_ids = [] # [{"key_ids":[]}, ...]   # (num_queries, "key_ids":(n_total_hit_keys))
    for query_id, _ in query2tag_ids:
        combined_key_ids = []
        for tag_ids in _["tag_ids"]:
            tag_dict = get_tag_dict(tag_ids, tag2key)
            key_ids = tag_dict["key_ids"]
            combined_key_ids += key_ids
        
        query2key_ids.append({"key_ids":combined_key_ids})
        selected_keys = [keys[key_id] for key_id in combined_key_ids]
        requests.append(search_engine(queries[query_id], selected_keys, k=k_key, return_relevance=True))
    
    output = await asyncio.gather(*requests) # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]
    
    return output

DEFAULT_KEYS = []
for i in range(len(df)):
    key = f"Sentence:'''{df.iloc[i]["text"]}'''"
    DEFAULT_KEYS.append(key)

all_questions = []
correct_ids = []
for id in query_dict:
    all_questions += query_dict[id]
    correct_ids += [id for _ in range(len(question_dict[id]))]
    
#output = await rmsearch2(queries=all_questions, k=20)
output = search_key(all_questions, DEFAULT_KEYS, tag2key, k_tag=2, k_key=5)
#output = await search_engine(all_questions, DEFAULT_KEYS, k=20, return_relevance=True)  # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]

for i in range(len(output)):
    output[i]["correct_id"] = correct_ids[i]
    for j in range(len(output[i]["keys"])):
        output[i]["keys"][j]["relevant_id"] = output[i]["keys"][j]["key_id"]

with open("sentences_relevant_to_questions.json", "w") as f:
    json.dump(output, f)

print("file_saved")


In [ ]:
# Judge which sentence is more relevant to a question

from itertools import combinations
import random

prompts = []
all_requests = AllRequests(max_request=40)

import os, re, json

with open("file_dict.json") as f:
    cs_file_dict = json.load(f)

with open("question_dict.json") as f:
    question_dict = json.load(f)

with open("files_relevant_to_questions.json") as f:
    files_relevant_to_questions = json.load(f)  # [{"query":, "correct_file_path": , "keys":[{"file_path":,}]}, ]

file_paths = list(cs_file_dict)

print("len(files_relevant_to_questions): ", len(files_relevant_to_questions))

for file_dict in files_relevant_to_questions:
    question = file_dict["query"]
    file_paths = [key_dict["file_path"] for key_dict in file_dict["keys"]]

    file_paths = file_paths[:8]
    
    file_path_pairs = list(combinations(file_paths, 2))  # [(path1, path2), ...]
    sample_file_path_pairs = random.sample(file_path_pairs, 5)

    for request_id, pair in enumerate(sample_file_path_pairs):
    
        file_path1 = pair[0]
        file_path2 = pair[1]
    
        system = f"""You are a brilliant judge who decides which text is more relevant to a given query. 
You will be given a query, 2 sentences.
Please carefully analyze these two sentences and then return your answer following the output format.

Output format:
<ID> 1 or 2 (file id more relevant to given query) </ID>"""

        user_prompt = f"""Query: {question}

Sentence 1:
'''{cs_file_dict[file_path1]}'''

Sentence 2:
'''{cs_file_dict[file_path2]}'''

Instructions:
1. Analyze content of each file.
2. Think which file is more relevant to query.
2. Please return id of the more relevant file enclosing it within <ID></ID> tag.

Let's think step by step following each step of the instructions."""


        prompt = tokenizer.apply_chat_template(
                    [{"role": "system", "content":system}, {"role": "user", "content":user_prompt}],
                    tokenize=False,
                    add_generation_prompt=True
                )

    
        request = {"request_id":request_id, "prompt":prompt, "file_paths":[file_path1, file_path2], "question":question}
        all_requests.add(request)

print("len(all_requests.requests): ", len(all_requests.requests))

results = await all_requests.process(max_tokens = 3000, temperature=0, save_dir = "relevant_file_progress5", restart = False)


## Embedding Model Gets TopN-Relevant Sentences and LLM Judges Which Content Is More Relevant to A Question

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sentence_transformers.quantization import quantize_embeddings

# 1. Specify preffered dimensions
dimensions = 512

# 2. load model
model = SentenceTransformer("intfloat/e5-mistral-7b-instruct").to("cuda")

'''
query_embedding = model.encode(advices, prompt_name="query")
# Equivalent Alternatives:
# query_embedding = model.encode(query_prompt + query)
# query_embedding = model.encode(query, prompt=query_prompt)

docs_embeddings = model.encode(advices)

# Optional: Quantize the embeddings
binary_query_embedding = quantize_embeddings(query_embedding, precision="ubinary")
binary_query_embedding = torch.tensor(binary_query_embedding).to("cuda")
binary_docs_embeddings = quantize_embeddings(docs_embeddings, precision="ubinary")
binary_docs_embeddings = torch.tensor(binary_docs_embeddings).to("cuda")

advice_similarities = cos_sim(query_embedding, docs_embeddings)
torch.save(advice_similarities, f"{exp_dir}/advice_similarities.pt")
'''


In [ ]:
import pandas as pd
import json

save_name = "smollm-corpus"
log_save_dir = "progress_search_log"
df = pd.read_csv(f"./data/{save_name}/df_small.csv")
with open(f"./data/{save_name}/query_dict.json") as f:
    query_dict = json.load(f)  # {"id":{"titles":[], "keywords":[], "questions":[], "irr_questions":[]}, ...}

k_key = 100

In [ ]:
import os
import json
import torch

def _extract_embedding(record):
    # Tries common shapes; returns 1D CPU float tensor
    if hasattr(record, "outputs") and hasattr(record.outputs, "embedding"):
        vec = record.outputs.embedding
    elif hasattr(record, "embedding"):
        vec = record.embedding
    else:
        vec = record
    return torch.as_tensor(vec, dtype=torch.float32).detach().cpu()


@torch.inference_mode()
def _embed_with_alignment(
    texts,
    model,
    batch_size=32,
    verbose=True,
    save_path=None,
    restart=False,
):
    """
    Embeds arbitrary 'texts' (list). Returns:
      E        : [N, D] float32 CPU tensor (zeros for bad/failed items)
      bad_mask: [N] bool, True where item failed (bad format/exception/dim mismatch)
    Also:
      - After each processed batch, checkpoint to `save_path` (if provided).
      - If `restart=True` and a checkpoint exists, resume from where it left off.
    """
    assert isinstance(texts, (list, tuple)), "texts must be a list/tuple"
    N = len(texts)
    bad_mask = torch.zeros(N, dtype=torch.bool)
    done_mask = torch.zeros(N, dtype=torch.bool)
    D = None
    E = None  # Will become [N, D] once D is known

    def log(msg):
        if verbose:
            print(msg)

    # Ensure dir exists
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Try to resume
    if restart and save_path is not None and os.path.exists(save_path):
        try:
            state = torch.load(save_path, map_location="cpu")
            if int(state.get("N", -1)) == N:
                D = state.get("D", None)
                bad_mask = state.get("bad_mask", bad_mask)
                done_mask = state.get("done_mask", done_mask)
                E = state.get("E", None)
                log(f"[resume] Loaded checkpoint: {save_path} (done {int(done_mask.sum())}/{N})")
            else:
                log(f"[resume] Checkpoint N mismatch (ckpt={state.get('N')} vs now={N}); starting fresh.")
        except Exception as e:
            log(f"[resume] Failed to load checkpoint ({e}); starting fresh.")

    def save_checkpoint():
        if save_path is None:
            return
        tmp_path = save_path + ".tmp"
        state = {
            "version": 1,
            "N": N,
            "D": D,
            "E": E,                # may be None until first good embedding
            "bad_mask": bad_mask,
            "done_mask": done_mask,
        }
        torch.save(state, tmp_path)
        os.replace(tmp_path, save_path)  # atomic on POSIX

    # Helper to ensure E allocated when D is known for the first time
    def ensure_E(dim):
        nonlocal D, E
        if D is None:
            D = int(dim)
        if E is None:
            E = torch.zeros((N, D), dtype=torch.float32)  # CPU

    # Indices that still need work
    def remaining_indices():
        return torch.nonzero(~done_mask).view(-1).tolist()

    todo = remaining_indices()
    while len(todo) > 0:
        idxs = todo[:batch_size]
        batch = [texts[i] for i in idxs]
        try:
            outputs = model.encode(batch)
            outs = list(outputs) if not isinstance(outputs, list) else outputs

            for i, out in zip(idxs, outs):
                try:
                    emb = _extract_embedding(out)
                    ensure_E(emb.shape[-1])
                    if emb.ndim != 1 or emb.shape[-1] != D:
                        raise ValueError(f"bad dim {tuple(emb.shape)} vs {D}")
                    E[i] = emb
                    done_mask[i] = True
                except Exception as e_item:
                    bad_mask[i] = True
                    done_mask[i] = True
                    log(f"[mark bad] idx={i}: {e_item}")

        except Exception as e_batch:
            log(f"[fallback per-item] {e_batch}")
            for i in idxs:
                try:
                    single = model.encode([texts[i]])
                    out = single[0] if hasattr(single, "__getitem__") else single
                    emb = _extract_embedding(out)
                    ensure_E(emb.shape[-1])
                    if emb.ndim != 1 or emb.shape[-1] != D:
                        raise ValueError(f"bad dim {tuple(emb.shape)} vs {D}")
                    E[i] = emb
                    done_mask[i] = True
                except Exception as e_item:
                    bad_mask[i] = True
                    done_mask[i] = True
                    log(f"[mark bad] idx={i}: {e_item}")

        # Save after each batch
        save_checkpoint()
        log(f"{done_mask.sum().item()} / {N} embedded")
        todo = remaining_indices()

    # If never got any valid embedding, set D=1 and build zeros
    if D is None or E is None:
        D = 1
        E = torch.zeros((N, D), dtype=torch.float32)

    return E, bad_mask


def search_key(
    queries,
    keys,
    k_key=5,
    device_compute=None,
    empty_cache=False,
    verbose=True,
    save_name="run",
    restart=False,
    query_batch_size=5000,
    key_batch_size=20,
):
    """
    Embeds queries/keys with checkpointing and resume support.
    Saves after every batch to:
      ./data/{save_name}/query_embed.pt
      ./data/{save_name}/key_embed.pt

    Returns:
      rel_values: [Q, topk]
      key_ids   : [Q, topk] (long)
    """
    if device_compute is None:
        device_compute = "cuda" if torch.cuda.is_available() else "cpu"
    device_compute = torch.device(device_compute)

    outdir = os.path.join("./data", save_name)
    os.makedirs(outdir, exist_ok=True)
    query_ckpt = os.path.join(outdir, "query_embed.pt")
    key_ckpt   = os.path.join(outdir, "key_embed.pt")

    print("embedding queries ...")
    queries_embed_cpu, bad_q = _embed_with_alignment(
        queries, model,
        batch_size=query_batch_size,
        verbose=verbose,
        save_path=query_ckpt,
        restart=restart,
    )
    print("embedding keys ...")
    keys_embed_cpu, bad_k = _embed_with_alignment(
        keys, model,
        batch_size=key_batch_size,
        verbose=verbose,
        save_path=key_ckpt,
        restart=restart,
    )

    Q, Dq = queries_embed_cpu.shape
    K, Dk = keys_embed_cpu.shape

    if Q == 0 or K == 0:
        return torch.empty((Q, 0)), torch.empty((Q, 0), dtype=torch.long)

    if Dq != Dk:
        raise ValueError(f"Embedding dimensions differ: queries D={Dq}, keys D={Dk}")

    # Move to compute device
    q = queries_embed_cpu.to(device_compute, non_blocking=True)
    k = keys_embed_cpu.to(device_compute, non_blocking=True)

    # Similarity
    relevance = q @ k.T  # [Q, K]

    # Set -inf where inputs were bad
    if bad_q.any():
        relevance[bad_q.to(relevance.device), :] = float("-inf")
    if bad_k.any():
        relevance[:, bad_k.to(relevance.device)] = float("-inf")

    # Top-k
    topk = min(k_key, K)
    if topk <= 0:
        return torch.full((Q, 0), float("-inf")), torch.empty((Q, 0), dtype=torch.long)

    rel_values, key_ids = torch.topk(relevance, k=topk, dim=1)

    # Return on CPU
    rel_values = rel_values.detach().cpu()
    key_ids = key_ids.detach().cpu()

    # Cleanup
    del q, k, relevance
    if empty_cache and device_compute.type == "cuda":
        torch.cuda.empty_cache()

    return rel_values, key_ids



KEYS = []
over_tokens = 0
for i in range(len(df)):
    if len(df.iloc[i]["text"]) > 15000:
        over_tokens += 1
    if type(df.iloc[i]["text"]) != str:
        print("Type Error: ", type(df.iloc[i]["text"]))
    KEYS.append(df.iloc[i]["text"])

queries = []
correct_ids = []
for id in query_dict:
    queries += query_dict[id]["questions"]
    correct_ids += [id for _ in range(len(query_dict[id]["questions"]))]

print("len(queries): ", len(queries))
print("len(KEYS): ", len(KEYS))
print("over_tokens: ", over_tokens)

k_key = 100
#relevance_values, key_ids = search_key(queries, KEYS, k_key=k_key)   # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]
# relevance_values: [n_queries, k_key],   key_ids: [n_queries, k_key]
# Make sure `model` exists (global or pass in if you adapt the function)

relevance_values, key_ids = search_key(
    queries,
    KEYS,
    k_key=k_key,
    save_name=save_name,
    restart=True,            # set True to resume from existing pt files
    query_batch_size=5000,   # tune as you like
    key_batch_size=20,       # tune as you like
)


output = []
for query_id in range(len(key_ids)):
    output_dict = {"query":queries[query_id], "correct_id":correct_ids[query_id], "query_id":query_id}
    key_dict_list = []
    for j in range(len(key_ids[query_id])):
        key_id = key_ids[query_id][j].item()
        key_dict_list.append({"key": KEYS[key_id], "key_id":key_id})
        
    output_dict["keys"] = key_dict_list
    output.append(output_dict)

# output : [{"query":, "correct_id":, "query_id":, "keys":[{"key":, "key_id":}, ...]}, ...]

with open(f"./data/{save_name}/sentences_relevant_to_questions.json", "w") as f:
    json.dump(output, f)

print("file_saved")

### Old

In [ ]:
import torch

def _extract_embedding(record):
    # Tries common shapes; returns 1D CPU float tensor
    if hasattr(record, "outputs") and hasattr(record.outputs, "embedding"):
        vec = record.outputs.embedding
    elif hasattr(record, "embedding"):
        vec = record.embedding
    else:
        vec = record
    return torch.as_tensor(vec, dtype=torch.float32).detach().cpu()


@torch.inference_mode()
def _embed_with_alignment(texts, model, batch_size=32, verbose=True):
    """
    Returns:
      E: [N, D] CPU tensor (zeros for bad items)
      bad_mask: [N] bool, True where item failed (bad format/exception/dim mismatch)
    Preserves original order and length.
    """
    N = len(texts)
    embs = [None] * N
    bad_mask = torch.zeros(N, dtype=torch.bool)
    D = None

    def log(msg):
        if verbose:
            print(msg)

    for start in range(0, N, batch_size):
        idxs = list(range(start, min(start + batch_size, N)))
        batch = [texts[i] for i in idxs]
        try:
            #if start == 0: continue
            #model.log_requests = False
            outputs = model.encode(batch)
            print(f"{min(start + batch_size, N)} / {N} Finished")
            outs = list(outputs) if not isinstance(outputs, list) else outputs
            for i, out in zip(idxs, outs):
                try:
                    emb = _extract_embedding(out)
                    if D is None:
                        D = emb.shape[-1]
                    if emb.ndim != 1 or emb.shape[-1] != D:
                        raise ValueError(f"bad dim {tuple(emb.shape)} vs {D}")
                    embs[i] = emb
                except Exception as e:
                    bad_mask[i] = True
                    log(f"[mark bad] idx={i}: {e}")
        except Exception as e_batch:
            print("error happened", start)
            log(f"[fallback per-item] {e_batch}")
            for i, txt in zip(idxs, batch):
                try:
                    single = model.encode([txt])
                    out = single[0] if hasattr(single, "__getitem__") else single
                    emb = _extract_embedding(out)
                    if D is None:
                        D = emb.shape[-1]
                    if emb.ndim != 1 or emb.shape[-1] != D:
                        raise ValueError(f"bad dim {tuple(emb.shape)} vs {D}")
                    embs[i] = emb
                except Exception as e_item:
                    bad_mask[i] = True
                    log(f"[mark bad] idx={i}: {e_item}")

    if D is None:
        # No valid embeddings at all; use dummy dimension
        D = 1

    E = torch.zeros((N, D), dtype=torch.float32)  # CPU
    for i, emb in enumerate(embs):
        if emb is None or emb.shape[-1] != D:
            bad_mask[i] = True
        else:
            E[i] = emb

    return E, bad_mask


def search_key(queries, keys, k_key=5, batch_size=5000, device_compute=None, empty_cache=False, verbose=True):
    """
    Embeds queries/keys in batches (CPU), computes similarity on `device_compute`,
    and sets -inf in the relevance matrix for any bad query (row) or bad key (column).
    """
    if device_compute is None:
        device_compute = "cuda" if torch.cuda.is_available() else "cpu"
    device_compute = torch.device(device_compute)

    print("embedding queries ...")
    queries_embed_cpu, bad_q = _embed_with_alignment(queries, model, batch_size=5000, verbose=verbose)
    print("embedding keys ...")
    keys_embed_cpu, bad_k = _embed_with_alignment(keys, model, batch_size=20, verbose=verbose)

    Q, Dq = queries_embed_cpu.shape
    K, Dk = keys_embed_cpu.shape

    if Q == 0 or K == 0:
        # Nothing to compare
        return torch.empty((Q, 0)), torch.empty((Q, 0), dtype=torch.long)

    if Dq != Dk:
        raise ValueError(f"Embedding dimensions differ: queries D={Dq}, keys D={Dk}")

    # Move to compute device
    q = queries_embed_cpu.to(device_compute, non_blocking=True)
    k = keys_embed_cpu.to(device_compute, non_blocking=True)

    # Similarity
    relevance = q @ k.T  # [Q, K]

    # Set -inf where inputs were bad
    if bad_q.any():
        relevance[bad_q.to(relevance.device), :] = float("-inf")
    if bad_k.any():
        relevance[:, bad_k.to(relevance.device)] = float("-inf")

    # Top-k
    topk = min(k_key, K)
    if topk <= 0:
        return torch.full((Q, 0), float("-inf")), torch.empty((Q, 0), dtype=torch.long)

    rel_values, key_ids = torch.topk(relevance, k=topk, dim=1)

    # Return on CPU
    rel_values = rel_values.detach().cpu()
    key_ids = key_ids.detach().cpu()

    # Cleanup
    del q, k, relevance
    if empty_cache and device_compute.type == "cuda":
        torch.cuda.empty_cache()

    return rel_values, key_ids

'''

def search_key(queries, keys, k_key=5):

    print("embedding queries ...")
    outputs = model.embed(queries)
    queries_embed = []
    for prompt, output in zip(prompts, outputs):
        queries_embed.append(output.outputs.embedding)
    queries_embed = torch.tensor(queries_embed).detach().cpu()
    del outputs
    print("query embedding done")
    
    print("embedding keys ...")
    outputs = model.embed(keys[:500])
    keys_embed = []
    for prompt, output in zip(prompts, outputs):
        keys_embed.append(output.outputs.embedding)
    keys_embed = torch.tensor(keys_embed).detach().cpu()
    del outputs
    print("key embedding done")

    relevance = torch.matmul(queries_embed, keys_embed.transpose(0,1))
    rel_values, key_ids = torch.topk(relevance, k = k_key)
    
    return rel_values, key_ids

'''

KEYS = []
over_tokens = 0
for i in range(len(df)):
    if len(df.iloc[i]["text"]) > 15000:
        over_tokens += 1
    if type(df.iloc[i]["text"]) != str:
        print("Type Error: ", type(df.iloc[i]["text"]))
    KEYS.append(df.iloc[i]["text"])

queries = []
correct_ids = []
for id in query_dict:
    queries += query_dict[id]["questions"]
    correct_ids += [id for _ in range(len(query_dict[id]["questions"]))]

print("len(queries): ", len(queries))
print("len(KEYS): ", len(KEYS))
print("over_tokens: ", over_tokens)
    
relevance_values, key_ids = search_key(queries, KEYS, k_key=k_key)   # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]
# relevance_values: [n_queries, k_key],   key_ids: [n_queries, k_key]


output = []
for query_id in range(len(key_ids)):
    output_dict = {"query":queries[query_id], "correct_id":correct_ids[query_id], "query_id":query_id}
    key_dict_list = []
    for j in range(len(key_ids[query_id])):
        key_id = key_ids[query_id][j].item()
        key_dict_list.append({"key": KEYS[key_id], "key_id":key_ids[key_id]})
        
    output_dict["keys"] = key_dict_list
    output.append(output_dict)

with open(f"./data/{save_name}/sentences_relevant_to_questions.json", "w") as f:
    json.dump(output, f)

print("file_saved")



In [ ]:
# Use Model from local environment

import asyncio
from transformers import AutoTokenizer
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams, PoolingParams
import time, warnings
import os
from tqdm import tqdm

#model_name = "/workspace/qwen7b"
model_name = "intfloat/e5-mistral-7b-instruct"
tensor_parallel_size = 1
pipeline_parallel_size = 1

engine_args = AsyncEngineArgs(
    model = model_name,
    task="embed",
    tensor_parallel_size = tensor_parallel_size,
    pipeline_parallel_size = pipeline_parallel_size,
    gpu_memory_utilization=0.95,
    disable_log_stats=True,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)
engine.log_requests = False
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

class AllRequests:
    
    def __init__(self, max_request):
        self.max_request = max_request
        self.requests = []
        self.request_ids = []
        self.request_id = 0
        self.results = []
        self.finished_ids = []
        self.progress_bar = None
        
    def add(self, request):
        self.requests.append(request)
        self.request_ids.append(self.request_id)
        self.request_id += 1
    
    async def process(self, model=model_name, max_tokens = 3000, temperature=0.4, save_dir = "progress_log", restart = False):

        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        if restart:
            if os.path.exists(f"{save_dir}/finished_ids.json") and os.path.exists(f"{save_dir}/results.json"):
                with open(f"{save_dir}/finished_ids.json") as f:
                    finished_ids = json.load(f)
                with open(f"{save_dir}/results.json") as f:
                    self.results = json.load(f)
                for finished_id in finished_ids:
                    self.finished_ids.append(finished_id)
                    id = self.request_ids.index(finished_id)
                    self.request_ids.pop(id)
                    self.requests.pop(id)
        else:
            os.remove(f"{save_dir}/finished_ids.json")
            os.remove(f"{save_dir}/results.json")

        total = len(self.requests)
        self.progress_bar = tqdm(total=total, desc="Processing Requests")

        await asyncio.gather(
            *[self.process_requests(temperature = temperature, max_tokens = max_tokens, save_dir=save_dir) for _ in range(self.max_request)]
        )

        self.progress_bar.close()
            
        return self.results


    async def process_requests(self, max_tokens = 3000, temperature=0.4, save_dir = "progress_log"):

        while len(self.requests) != 0:
            request_dict = self.requests.pop(0)
            request_id = self.request_ids.pop(0)

            prompt = request_dict["prompt"]

            final_output = None
            results_generator = Search.engine.encode(prompt, 
                                    PoolingParams(), 
                                    request_id
                                    )
            #results_generator = engine.generate(prompt, SamplingParams(temperature=temperature, max_tokens=max_tokens), request_id)
            async for request_output in results_generator:
                # print(request_output) => for streaming
                final_output = request_output

            embedding = final_output.outputs.data
            embedding = torch.tensor(embedding).float()
            #output = final_output.outputs[0].text
            
            request_dict["embedding"] = embedding
            self.results.append(request_dict)
            self.finished_ids.append(request_id)

            with open(f"{save_dir}/results.json", "w") as f:
                json.dump(self.results, f)
            with open(f"{save_dir}/finished_ids.json", "w") as f:
                json.dump(self.finished_ids, f)

            self.progress_bar.update(1)
            

In [ ]:
# Create an LLM.
# You should pass task="embed" for embedding models
model = LLM(
    model="intfloat/e5-mistral-7b-instruct",
    task="embed",
    enforce_eager=True,
    #disable_log_stats=True,
)

## Judge which sentence is more relevant to a question

In [ ]:
## Judge which sentence is more relevant to a question

from itertools import combinations
import random

prompts = []
all_requests = AllRequests(max_request=40)

import pandas as pd
import json

print("loading started")

save_name = "smollm-corpus"
log_save_dir = "progress_search_log"
df = pd.read_csv(f"./data/{save_name}/df_small.csv")
with open(f"./data/{save_name}/query_dict.json") as f:
    query_dict = json.load(f)  
with open(f"./data/{save_name}/sentences_relevant_to_questions.json") as f:
    relevant_sentences = json.load(f)

print("len(relevant_sentences): ", len(relevant_sentences))

# query_dict : {"id":{"titles":[], "keywords":[], "questions":[], "irr_questions":[]}, ...}
# relevant_sentences : [{"query":, "correct_id":, "query_id":, "keys":[{"key":, "key_id":}, ...]}, ...]

for sentence_dict in relevant_sentences:
    query_id = sentence_dict["query_id"]            # id of query, which is also the id in relevant_sentences
    correct_id = sentence_dict["correct_id"]        # id of sentence which query is made from
    query = sentence_dict["query"]
    sentence_ids = [key_dict["key_id"] for key_dict in sentence_dict["keys"]]
    sentences = [key_dict["key"] for key_dict in sentence_dict["keys"]]

    sentence_ids = sentence_ids[:3]
    if correct_id in sentence_ids[:2]:
        pass
    else:
        sentence_ids = sentence_ids[:2] + [int(correct_id)]
    
    sentence_id_pairs = list(combinations(sentence_ids, 2))  # [(path1, path2), ...]
    sample_sentence_id_pairs = random.sample(sentence_id_pairs, 1)

    for request_id, pair in enumerate(sample_sentence_id_pairs):
    
        sentence_id1 = pair[0]
        sentence_id2 = pair[1]
    
        system = f"""You are a brilliant judge who decides which text is more relevant to a given query. 
You will be given a query, 2 sentences.
Please carefully analyze these two sentences and then return your answer following the output format.

Output format:
<ID> 1 or 2 (file id more relevant to given query) </ID>"""

        user_prompt = f"""Query: {query}

Sentence 1:
'''{df.iloc[sentence_id1]["text"]}'''

Sentence 2:
'''{df.iloc[sentence_id2]["text"]}'''

Instructions:
1. Analyze content of each sentence.
2. Think which sentence is more relevant to the query.
3. Please return id of the more relevant sentence enclosing it within <ID></ID> tag.
4. If both sentences are highly related to the query or either of sentence is totally irrelevant to it, return -1 in the <ID></ID> tag.

Let's think step by step following each step of the instructions."""


        prompt = tokenizer.apply_chat_template(
                    [{"role": "system", "content":system}, {"role": "user", "content":user_prompt}],
                    tokenize=False,
                    add_generation_prompt=True
                )

    
        request = {"request_id":request_id, "prompt":prompt, "sentence_ids":[sentence_id1, sentence_id2], "question":query}
        all_requests.add(request)

print("len(all_requests.requests): ", len(all_requests.requests))

results = await all_requests.process(max_tokens = 3000, temperature=0, save_dir = "relevant_file_progress7", restart = True)


In [ ]:
print(len(output))
with open("relevant_file_progress6/finished_ids.json") as f:
    finished_ids = json.load(f)
print(len(finished_ids))

## Make dataset_list

In [ ]:
import json
with open("relevant_file_progress6/results.json") as f:
    results = json.load(f)

In [ ]:
import re
def extract_text(textC: str, textB: str) -> str | None:
    # Build a non-greedy pattern like r'<tag>(.*?)</tag>'
    pattern = rf"<{re.escape(textB)}>(.*?)</{re.escape(textB)}>"
    match = re.search(pattern, textC, flags=re.DOTALL)
    return match.group(1) if match else None

def extract_int(text: str) -> int | None:
    """
    Return the first integer found in `text`.
    If no digits appear, return None.
    """
    match = re.search(r"-?\d+", text)  # handles negative numbers too
    return int(match.group()) if match else None


In [ ]:
import json

save_name = "smollm-corpus"
log_save_dir = "progress_search_log"
df = pd.read_csv(f"./data/{save_name}/df_small.csv")


with open(f"./data/{save_name}/query_dict.json") as f:
    query_dict = json.load(f)  
with open(f"./data/{save_name}/sentences_relevant_to_questions.json") as f:
    relevant_sentences = json.load(f)

dataset_list = []
    
for result in results:
    request_id = result["request_id"]
    output = result["output"]
    sentence_ids = result["sentence_ids"]
    question = result["question"]
    id = extract_text(output, "ID")

    #print()
    #print("id: ", id)

    try:
        id = int(id)

    except:
        try:
            id = extract_int(output[-10:])
            
        except:
            continue

    if not (id == 1 or id == 2):
        continue
    else:
        if id == 1:
            chosen_sentence_id = sentence_ids[0]
            rejected_sentence_id = sentence_ids[1]
        else:
            chosen_sentence_id = sentence_ids[1]
            rejected_sentence_id = sentence_ids[0]

    dataset_list.append({
        "chosen_msg":[{'role': 'user', 'content':f"Give me relevant score between query and sentence;\n\nQuery:{question}\n\nSentence:```{df.iloc[chosen_sentence_id]['text']}```"}],
        "rejected_msg":[{'role': 'user', 'content':f"Give me relevant score between query and sentence;\n\nQuery:{question}\n\nSentence:```{df.iloc[rejected_sentence_id]['text']}```"}],
        "chosen_sentence_id":chosen_sentence_id,
        "rejected_sentence_id":rejected_sentence_id,
    })
    

#print(data_dict)
#with open("data_dict.json", "w") as f:
#    json.dump(data_dict, f)

exp_dir = "/workspace/RMS_exp/exp2"

if not os.path.exists(exp_dir):
    os.makedirs(exp_dir)

dataset_list_save_path = f"{exp_dir}/dataset_list.json"

with open(dataset_list_save_path, "w") as f:
    json.dump(dataset_list, f)



# Train

In [ ]:
!pip install ../RMSearch
#!pip3 install fastapi uvicorn[standard]

In [ ]:
from rmsearch import RMTrainer

model_name = "/workspace/llama3b-rm"
num_gpus = 2

rmtrainer = RMTrainer(model_name = model_name, num_gpus = num_gpus)

In [ ]:
exp_dir = "/workspace/RMS_exp/exp2"
dataset_list_save_path = f"{exp_dir}/dataset_list.json"
#dataset_save_path = f"{exp_dir}/dataset"
#train_ids_save_path = f"{exp_dir}/train_ids.json"
#test_ids_save_path = f"{exp_dir}/test_ids.json"
test_size = 100
#test_ratio = 0.1

import json
with open(dataset_list_save_path) as f:
    dataset_list = json.load(f)

tokenizer = rmtrainer.tokenizer

def formatting_func(examples):
    kwargs = {"padding": "max_length", "truncation": True, "max_length": 4000, "return_tensors": "pt", "add_special_tokens":False}
    chosen_msg = examples['chosen_msg']
    rejected_msg = examples['rejected_msg']

    if len(chosen_msg[0]["content"]) > 4000:
        chosen_msg[0]["content"] = chosen_msg[0]["content"][:4000] + "..."
    if len(rejected_msg[0]["content"]) > 4000:
        rejected_msg[0]["content"] = rejected_msg[0]["content"][:4000] + "..."

    prompt_plus_chosen_response = tokenizer.apply_chat_template(chosen_msg, tokenize=False)
    prompt_plus_rejected_response = tokenizer.apply_chat_template(rejected_msg, tokenize=False)

    tokens_chosen = tokenizer.encode_plus(prompt_plus_chosen_response, **kwargs)
    tokens_rejected = tokenizer.encode_plus(prompt_plus_rejected_response, **kwargs)
    
    return {
        "input_ids_chosen": tokens_chosen["input_ids"][0], "attention_mask_chosen": tokens_chosen["attention_mask"][0],
        "input_ids_rejected": tokens_rejected["input_ids"][0], "attention_mask_rejected": tokens_rejected["attention_mask"][0]
    }

'''
def formatting_func(examples):
    # examples : sentence1, sentence2, score, 
    kwargs = {"padding": "max_length", "truncation": True, "max_length": 4000, "return_tensors": "pt", "add_special_tokens":False}

    sentence1 = examples["sentence1"]
    sentence2 = examples["sentence2"]
    
    if len(sentence1) > 3000:
        sentence1 = sentence1[:3000] + "..."
    if len(sentence2) > 3000:
        sentence2 = sentence2[:3000] + "..."
    
    msg = [{"role":"user", "content":f"Relevance between\n\n<sentence1>{sentence1}</sentence1>\n\nand\n\n<sentence2>{sentence2}</sentence2>"}]
    
    prompt_plus_response = tokenizer.apply_chat_template(msg, tokenize=False)

    tokens = tokenizer.encode_plus(prompt_plus_response, **kwargs)
    
    return {
        "input_ids": tokens["input_ids"][0], "attention_mask": tokens["attention_mask"][0],
        "score": examples["score"]
    }
'''

formatted_dataset = rmtrainer.prepare_dataset(dataset_list, base_dir = exp_dir, test_size = test_size, formatting_func=formatting_func)


In [ ]:
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig, TaskType
import accelerate

batch_size_per_device = 3
eval_batch_size_per_device = 2
model_save_dir = f"{exp_dir}/model1"

class CustomRewardTrainer(RewardTrainer):
    _tag_names = ["trl", "reward-trainer"]

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def train(self, *args, **kwargs): # You need this because it will use RewardTrainer compute_loss method without this. To use a subclass function, some method in the subclass must be called from main directly. 
        return super().train(*args, **kwargs)

    def evaluate(self, *args, **kwargs):
        return super().evaluate(num_print_samples=1, *args, **kwargs)

training_args = RewardConfig(
    output_dir=model_save_dir,
    per_device_train_batch_size=batch_size_per_device,
    per_device_eval_batch_size=eval_batch_size_per_device,
    eval_strategy="steps",
    eval_steps=40,
    eval_on_start=True,
    save_steps=20,
    logging_steps=1,
    num_train_epochs = 50,
    report_to=None,
    remove_unused_columns=False,
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    target_modules=["k_proj","q_proj","o_proj", "v_proj","down_proj","gate_proj","up_proj",],
    layers_to_transform=[25,26,27],
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
)

rmtrainer.train(
    formatted_dataset,
    training_args = training_args,
    peft_config = peft_config,
    trainer_cls = CustomRewardTrainer,
    #data_collator = custom_data_collator, 
)


## for advanced DPO

In [ ]:
from trl import RewardConfig
from peft import LoraConfig, TaskType
from typing import Any, Optional, Union
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import Trainer, PreTrainedModel
from trl.trainer.utils import compute_accuracy
import torch.nn.functional as F

batch_size_per_device = 5
eval_batch_size_per_device = 5
model_save_dir = f"{exp_dir}/model1"

'''
from trl.data_utils import maybe_apply_chat_template
from trl.trainer.reward_config import RewardConfig
from trl.trainer.utils import (
    RewardDataCollatorWithPadding,
    compute_accuracy,
    decode_and_strip_padding,
    disable_dropout_in_model,
    generate_model_card,
    get_comet_experiment_url,
    log_table_to_comet_experiment,
    print_rich_table,
)
'''

#class CustomRewardTrainer(RewardTrainer):
class CustomRewardTrainer(Trainer):
    _tag_names = ["trl", "reward-trainer"]

    def __init__(self, *args, **kwargs):
        super().__init__(compute_metrics=compute_accuracy, *args, **kwargs)

    def compute_loss(
        self,
        model: Union[PreTrainedModel, nn.Module],
        inputs: dict[str, Union[torch.Tensor, Any]],
        return_outputs=False,
        num_items_in_batch=None,
    ) -> Union[torch.Tensor, tuple[torch.Tensor, dict[str, torch.Tensor]]]:

        #n_chosens = inputs["num_chosen"]  #[num_gpus]
        #n_rejecteds = inputs["num_rejected"]  #[num_gpus]
        #chosen_reject_similarities = inputs["chosen_reject_similarities"]  #[num_gpus, n_chosens, n_rejecteds]

        #print(inputs["input_ids_chosen"].shape)
        #print(inputs["input_ids_rejected"].shape)

        #print(inputs["input_ids"].shape)
        num_batch, max_length = inputs["input_ids"].shape
        input_ids = inputs["input_ids"] #.reshape(num_gpus*num_batch, max_length)
        attention_mask = inputs["attention_mask"] #.reshape(num_gpus*num_batch, max_length)
        score = inputs["score"] #.reshape(num_gpus*num_batch) 
        #print(inputs["input_ids"].shape)
        #attention_mask = inputs["attention_mask"].reshape(inputs["attention_mask"].shape[0]*inputs["attention_mask"].shape[1], inputs["attention_mask"].shape[2])

        
        all_rewards = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )["logits"]

        #print("score ", score.shape)
        #print("all_rewards ", all_rewards.shape)

        loss = F.mse_loss(all_rewards, score, reduction="mean")

        if return_outputs:
            return loss, {
                "all_rewards": all_rewards,
                "score": score,
            }
        return loss
    
        '''

        #print("1", all_rewards.shape)
        all_rewards = all_rewards.reshape(num_gpus, num_batch)
        #print("2", all_rewards.shape)
        
        all_chosen_idx = []
        all_rejected_idx = []
        total_loss = 0
        for i in range(num_gpus):
            rewards = all_rewards[i]
            
            n_chosen = n_chosens[i]
            n_rejected = n_rejecteds[i]
            n_rows = n_chosen * n_rejected
            n_cols = n_chosen + n_rejected
            chosen_idx = torch.arange(n_chosen).repeat_interleave(n_rejected)   # shape: [n_rows]
            rejected_idx = torch.arange(n_rejected).repeat(n_chosen)   # shape: [n_rows]
            all_chosen_idx.append(chosen_idx)
            all_rejected_idx.append(rejected_idx)
            
            # Create cr_matrix
            cr_matrix = torch.zeros(n_rows, n_cols)
            rows = torch.arange(n_rows)
            cr_matrix[rows, chosen_idx] = 1
            cr_matrix[rows, n_chosen + rejected_idx] = -1
            cr_matrix = cr_matrix.to(rewards.device)

            # Create coe_vector
            sims = chosen_reject_similarities[chosen_idx, rejected_idx] # shape: [n_rows]
            coe_vector = 1-sims

            print(coe_vector.shape)
            print(coe_vector)

            # Test cr_matrix
            test_tensor = torch.ones(rewards.shape).to(cr_matrix.device)
            result_tensor = torch.matmul(cr_matrix, test_tensor)
            #print("result_tensor.shape: ", result_tensor.shape)
            #print("result_tensor.mean() : ", result_tensor.mean())
            
            # calculate loss, optionally modulate with margin
            if "margin" in inputs:
                loss = -nn.functional.logsigmoid(coe_vector * torch.matmul(cr_matrix, rewards) - inputs["margin"]).mean()
            else:
                loss = -nn.functional.logsigmoid(coe_vector * torch.matmul(cr_matrix, rewards)).mean()

            total_loss += loss
            #all_losses.append(loss)

        #total_loss = torch.tensor(all_losses).mean()
        
        if return_outputs:
            return total_loss, {
                "all_rewards": all_rewards,
                "all_chosen_idx": all_chosen_idx,
                "all_rejected_idx": all_rejected_idx,
                "n_chosen":n_chosen,
                "n_rejected":n_rejected,
            }
        return total_loss

        '''

    def prediction_step(
        self,
        model: Union[PreTrainedModel, nn.Module],
        inputs: dict[str, Union[torch.Tensor, Any]],
        prediction_loss_only: bool,
        ignore_keys: Optional[list[str]] = None,
    ) -> tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
        
        inputs = self._prepare_inputs(inputs)
        if ignore_keys is None:
            if hasattr(self.model, "config"):
                ignore_keys = getattr(self.model.config, "keys_to_ignore_at_inference", [])
            else:
                ignore_keys = []

        with torch.no_grad():
            loss, logits_dict = self.compute_loss(model, inputs, return_outputs=True)

        return loss, None, None
    
        """
        '''
        chosen_rewards = logits_dict["chosen_rewards"]
        rejected_rewards = logits_dict["rejected_rewards"]
        chosen_idx = logits_dict["chosen_idx"]
        rejected_idx = logits_dict["rejected_idx"]
        '''

        all_rewards = logits_dict["all_rewards"]
        
        if prediction_loss_only:
            return (loss, None, None)

        loss = loss.detach()
        all_logits = torch.tensor([])
        for i in range(len(all_rewards)):
            rewards = all_rewards[i]
            #print("rewards.shape: ", rewards.shape)
            #print("all_chosen_idx[i]: ", all_chosen_idx[i])
            chosen_logits = rewards[all_chosen_idx[i]].unsqueeze(-1)
            rejected_logits = rewards[n_chosen + all_rejected_idx[i]].unsqueeze(-1)
            logits = torch.cat((chosen_logits, rejected_logits), dim=1)
            #print("logits.shape: ", logits.shape)
            all_logits = all_logits.to(logits.device)
            all_logits = torch.cat((all_logits, logits), dim=0)

        all_logits = all_logits.softmax(dim=1).detach()
        #print("all_logits: ", all_logits)
        #print("all_logits.shape: ", all_logits.shape)
        #logits = torch.cat((chosen_rewards[chosen_idx], rejected_rewards[rejected_idx]), dim=1)
        #logits = logits.softmax(dim=1).detach()
        '''
        logits = tuple(v for k, v in logits_dict.items() if k not in ignore_keys)
        logits = nested_detach(logits)
        # Stack accepted against rejected, mean over logits
        # and softmax to get preferences between accepted and rejected to sum to 1
        logits = torch.stack(logits).mean(dim=2).softmax(dim=0).T
        '''

        labels = torch.zeros(all_logits.shape[0])
        labels = self._prepare_inputs(labels)

        return loss, all_logits, labels
        """

    def train(self, *args, **kwargs): # You need this because it will use RewardTrainer compute_loss method without this. To use a subclass function, some method in the subclass must be called from main directly. 
        return super().train(*args, **kwargs)

    def evaluate(self, *args, **kwargs):
        return super().evaluate(*args, **kwargs)
        #return super(RewardTrainer, self).evaluate(*args, **kwargs)
        #return super().evaluate(num_print_samples=1, *args, **kwargs) # this fell in an error for some reason

def custom_data_collator(features):
    batch = {}
    
    # For fields that are tensors, we stack them.
    '''
    tensor_fields = [
        "input_ids", "attention_mask",
    ]
    '''
    tensor_fields = [
        "input_ids", "attention_mask",
    ]
    
    for field in tensor_fields:
        batch[field] = torch.stack([torch.tensor(f[field]) for f in features])  #[num_gpus, num_advice_per_batch, max_length]

    batch["score"] = torch.stack([torch.tensor(f["score"]) for f in features])  #[num_gpus, num_advice_per_batch]
    
    # For the original prompts (strings), we simply collect them in a list.
    #non_tensor_fields = ["num_chosen", "num_rejected", "problem_id"]
    #non_tensor_fields = ["score"]
    #for field in non_tensor_fields:
    #    batch[field] = [f[field] for f in features]
    
    return batch

training_args = RewardConfig(
    output_dir=model_save_dir,
    per_device_train_batch_size=batch_size_per_device,
    per_device_eval_batch_size=eval_batch_size_per_device,
    eval_strategy="steps",
    eval_steps=40,
    eval_on_start=True,
    save_steps=20,
    logging_steps=1,
    num_train_epochs = 50,
    report_to=None,
    remove_unused_columns=False,
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    target_modules=["k_proj","q_proj","o_proj", "v_proj","down_proj","gate_proj","up_proj",],
    layers_to_transform=[25,26,27],
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
)

rmtrainer.train(formatted_dataset,
                training_args = training_args,
                peft_config = peft_config,
                trainer_cls = CustomRewardTrainer,
                data_collator = custom_data_collator, 
               )



In [ ]:
exp_dir = "exp2"
param_save_path = "model1"
last_check_point = "checkpoint-1100"
train_data_path = f"{exp_dir}/{param_save_path}/{last_check_point}/trainer_state.json"

import json
with open(train_data_path) as f:
    train_data = json.load(f)
    log_hist = train_data["log_history"]

train_epoch = []
eval_epoch = []
train_loss = []
eval_loss = []
eval_acc = []
for i, log_hist_data in enumerate(log_hist):
    if "eval_loss" in log_hist_data:
        eval_epoch.append(log_hist_data["epoch"])
        eval_loss.append(log_hist_data["eval_loss"])
        eval_acc.append(log_hist_data["eval_accuracy"])
    else:
        train_epoch.append(log_hist_data["epoch"])
        train_loss.append(log_hist_data["loss"])

import matplotlib.pyplot as plt
# Create a new figure with a specific size
plt.figure(figsize=(8, 6))

# Plot each set of values with markers and labels
#plt.plot(train_epoch, train_loss, marker='o', label='train_loss')
plt.plot(eval_epoch, eval_loss, marker='s', label='eval_loss')
plt.plot(eval_epoch, eval_acc, marker='^', label='eval_acc')

# Add labels and title
plt.xlabel('Epoch')
plt.ylabel('Train Loss and Eval Loss')
plt.title('Search Engine Training')

# Enable the legend and grid
plt.legend()
plt.grid(True)

# Display the plot
plt.show()



# Model Conversion

In [ ]:
base_model_name = "/workspace/llama3b-rm"
checkpoint_path = "exp3/model1/checkpoint-40"
model_name = "exp3-model1-step40"

from transformers import AutoModelForSequenceClassification,AutoTokenizer,TrainingArguments,AutoConfig,AutoModelForCausalLM
from peft import PeftModel, PeftConfig

tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left",add_eos_token=False,add_bos_token=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    config.pad_token_id = config.eos_token_id

# Load the pre-trained model without the LM head.
# AutoModelForCausalLM usually refers to models with the LM head included, so you'd typically use a more specific base model class.
#base_model = AutoModelForCausalLM.from_config(config).base_model
model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=1)

lora_model = PeftModel.from_pretrained(model, checkpoint_path)

reward_model = lora_model.merge_and_unload()

reward_model

In [ ]:
save_dir = f"./{model_name}-converted-model"
score_save_path = f"./{model_name}-converted-score.pt"

import torch
tokenizer.save_pretrained(save_dir)
reward_model.save_pretrained(save_dir)
torch.save(reward_model.score.weight.data, score_save_path)
del reward_model

generate_model = AutoModelForCausalLM.from_pretrained(save_dir)
generate_model.save_pretrained(save_dir)
del generate_model

# Evaluation (Retrival)

## Embedding

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F

from vllm_embed import build_embedding_model, embed as embed_with_model


def _as_int(value, default):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _as_float(value, default=None):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


if not torch.cuda.is_available():
    raise RuntimeError("vLLM embedding requires CUDA; no GPU detected.")

device = torch.device("cuda")

working_dir = os.environ.get("RM_WORKDIR", "/workspace/RMS_exp")
data_name = os.environ.get("RM_DATASET", "smollm-corpus")
embed_model_name = os.environ.get("EMB_MODEL", "intfloat/e5-mistral-7b-instruct")
model_name = f"/workspace/llama3b-rm-converted-model"

exp_eval_dir = f"{model_name}-eval"
data_dir = Path(working_dir) / "data" / data_name
df = pd.read_csv(data_dir / "df_small.csv")
with open(data_dir / "query_dict.json") as f:
    query_dict = json.load(f)

sentences = [str(text) for text in df["text"].tolist()]
queries = []
correct_ids = []
for idx in range(len(df)):
    for question in query_dict.get(str(idx), {}).get("questions", []):
        question = (question or "").strip()
        if not question:
            continue
        queries.append(question)
        correct_ids.append(idx)

if not queries:
    raise ValueError("No queries found in query_dict.json")
if not sentences:
    raise ValueError("No keys found in df_small.csv")

tensor_parallel_size = max(
    1,
    _as_int(
        os.environ.get("EMB_TENSOR_PARALLEL_SIZE"),
        _as_int(globals().get("tensor_parallel_size"), 1),
    ),
)
num_instances = max(
    1,
    _as_int(
        os.environ.get("EMB_NUM_INSTANCES"),
        _as_int(globals().get("num_instances"), 1),
    ),
)
embed_batch_size = max(1, _as_int(os.environ.get("EMB_BATCH_SIZE"), 32))
score_batch_size = max(1, _as_int(os.environ.get("EMB_SCORE_BATCH"), 32))
timeout_value = os.environ.get("EMB_TIMEOUT")
timeout_s = _as_float(timeout_value) if timeout_value else None

candidate_groups = globals().get("device_groups")
device_groups = None
if isinstance(candidate_groups, list) and candidate_groups:
    if all(isinstance(group, list) and len(group) == tensor_parallel_size for group in candidate_groups):
        device_groups = candidate_groups[:num_instances]

if device_groups is None:
    total_gpus = torch.cuda.device_count()
    if total_gpus <= 0:
        raise RuntimeError("CUDA GPU required for vLLM embedding.")
    tensor_parallel_size = min(tensor_parallel_size, total_gpus)
    tensor_parallel_size = max(1, tensor_parallel_size)
    max_instances = max(1, total_gpus // tensor_parallel_size)
    num_instances = min(num_instances, max_instances)
    num_instances = max(1, num_instances)
    device_ids = list(range(total_gpus))
    device_groups = []
    offset = 0
    for _ in range(num_instances):
        group = device_ids[offset: offset + tensor_parallel_size]
        if len(group) < tensor_parallel_size:
            break
        device_groups.append(group)
        offset += tensor_parallel_size
    if not device_groups:
        device_groups = [device_ids[:tensor_parallel_size]]
    num_instances = len(device_groups)

embed_pool = build_embedding_model(
    embed_model_name,
    tensor_parallel_size=tensor_parallel_size,
    num_instances=num_instances,
    device_groups=device_groups,
    output_to_cpu=True,
    max_model_len=_as_int(os.environ.get("EMB_MAX_MODEL_LEN"), 4096),
)

try:
    key_vectors = embed_with_model(
        embed_pool,
        sentences,
        batch_size=embed_batch_size,
        timeout_s=timeout_s,
    )
    query_vectors = embed_with_model(
        embed_pool,
        queries,
        batch_size=embed_batch_size,
        timeout_s=timeout_s,
    )
finally:
    embed_pool.close()

if len(key_vectors) != len(sentences):
    raise RuntimeError("Embedding output size mismatch for keys.")
if len(query_vectors) != len(queries):
    raise RuntimeError("Embedding output size mismatch for queries.")

key_emb = torch.tensor(key_vectors, dtype=torch.float32, device=device)
query_emb = torch.tensor(query_vectors, dtype=torch.float32, device=device)

key_emb = F.normalize(key_emb, dim=-1)
query_emb = F.normalize(query_emb, dim=-1)

num_queries = query_emb.size(0)
num_keys = key_emb.size(0)
topk_default = _as_int(globals().get("k_key"), 100)
topk = max(1, min(num_keys, topk_default))

relevance_dict = []

with torch.inference_mode():
    key_emb_t = key_emb.transpose(0, 1).contiguous()
    for start in range(0, num_queries, score_batch_size):
        end = min(start + score_batch_size, num_queries)
        scores = torch.matmul(query_emb[start:end], key_emb_t)
        top_scores, top_indices = torch.topk(scores, k=topk, dim=-1)
        top_scores = top_scores.detach().cpu()
        top_indices = top_indices.detach().cpu()

        for local_idx in range(top_indices.size(0)):
            global_idx = start + local_idx
            entry = {
                "query": queries[global_idx],
                "query_id": global_idx,
                "correct_id": int(correct_ids[global_idx]),
                "keys": [],
            }
            for rank, (score, key_id) in enumerate(
                zip(top_scores[local_idx], top_indices[local_idx]),
                start=1,
            ):
                key_idx = int(key_id.item())
                entry["keys"].append(
                    {
                        "key": sentences[key_idx],
                        "key_id": key_idx,
                        "relevance": float(score.item()),
                        "relevant_id": key_idx,
                        "rank": rank,
                    }
                )
            relevance_dict.append(entry)
        print(f"Processed {min(end, num_queries)} / {num_queries} queries", flush=True)

exp_eval_dir = globals().get("exp_eval_dir")
if not exp_eval_dir:
    exp_eval_dir = os.path.join(working_dir, "data", )
os.makedirs(exp_eval_dir, exist_ok=True)
output_path = os.path.join(exp_eval_dir, "relevance_dict_with_embedding.json")
with open(output_path, "w") as f:
    json.dump(relevance_dict, f)

print(f"Saved embedding-based relevance scores to {output_path}")




In [ ]:
# relevance_dict_with_embedding.json was quite heavy so this is for saving lighter file

topk_default = _as_int(globals().get("k_key"), 10)
topk = max(1, min(num_keys, topk_default))

relevance_dict = []

with torch.inference_mode():
    key_emb_t = key_emb.transpose(0, 1).contiguous()
    for start in range(0, num_queries, score_batch_size):
        end = min(start + score_batch_size, num_queries)
        scores = torch.matmul(query_emb[start:end], key_emb_t)
        top_scores, top_indices = torch.topk(scores, k=topk, dim=-1)
        top_scores = top_scores.detach().cpu()
        top_indices = top_indices.detach().cpu()

        for local_idx in range(top_indices.size(0)):
            global_idx = start + local_idx
            entry = {
                #"query": queries[global_idx],
                "query_id": global_idx,
                "correct_id": int(correct_ids[global_idx]),
                "keys": [],
            }
            for rank, (score, key_id) in enumerate(
                zip(top_scores[local_idx], top_indices[local_idx]),
                start=1,
            ):
                key_idx = int(key_id.item())
                entry["keys"].append(
                    {
                        #"key": sentences[key_idx],
                        "key_id": key_idx,
                        "relevance": float(score.item()),
                        "relevant_id": key_idx,
                        "rank": rank,
                    }
                )
            relevance_dict.append(entry)
        print(f"Processed {min(end, num_queries)} / {num_queries} queries", flush=True)

exp_eval_dir = globals().get("exp_eval_dir")
if not exp_eval_dir:
    exp_eval_dir = os.path.join(working_dir, "embedding-eval")
os.makedirs(exp_eval_dir, exist_ok=True)
output_path = os.path.join(exp_eval_dir, "relevance_dict_with_embedding_light.json")
with open(output_path, "w") as f:
    json.dump(relevance_dict, f)

print(f"Saved embedding-based relevance scores to {output_path}")

In [ ]:
import os, json

working_dir = os.environ.get("RM_WORKDIR", "/workspace/RMS_exp")
exp_eval_dir = globals().get("exp_eval_dir")
if not exp_eval_dir:
    exp_eval_dir = os.path.join(working_dir, "embedding-eval")

output_path = os.path.join(exp_eval_dir, "relevance_dict_with_embedding.json")
with open(output_path) as f:
    relevance_dict = json.load(f)

for i in range(len(relevance_dict)):
    del relevance_dict[i]["query"]
    for key_dict in relevance_dict[i]["keys"]:
        del key_dict["key"]

output_path = os.path.join(exp_eval_dir, "relevance_dict_with_embedding_light.json")
with open(output_path, "w") as f:
    json.dump(relevance_dict, f)

## With Graph

In [ ]:
# Need to use vllm==0.10.1 version to use this model
!pip install -U vllm==0.10.1

In [ ]:
# Data Loading

working_dir = "/workspace/RMS_exp"
#exp_dir = f"{working_dir}/exp2"
model_name = f"/workspace/llama3b-rm-converted-model"
exp_eval_dir = f"{model_name}-eval"
#dataset_load_path = f"{exp_dir}/dataset"
data_name = "smollm-corpus"
tensor_parallel_size = 1
num_instances = 4
output_path = f"{exp_eval_dir}/relevance_dict.json"

import os
import pandas as pd
import json
import torch

if not os.path.exists(exp_eval_dir): os.makedirs(exp_eval_dir)

df = pd.read_csv(f"{working_dir}/data/{data_name}/df_small.csv")                            # df["text"] = ["sentence1", .... ]
#df = df[:100]
with open(f"{working_dir}/data/{data_name}/query_dict.json") as f:
    query_dict = json.load(f)                                                               # { "0": {"titles":["",...], "keywords":["",...], "questions":["",...], "irr_questions":["",...] } }
with open(f"{working_dir}/data/{data_name}/sentences_relevant_to_questions.json") as f:
    relevant_sentences = json.load(f)                                                       # [{"query":, "query_id":, "correct_id":, "keys":[{"key":, "key_id":,}, ...]}, ... ]
with open(f"{working_dir}/data/{data_name}/tag2query.json") as f:
    tag_dict = json.load(f)                                                                 # [{"tag":"", "query_ids":[], "children":[{"tag":"", "query_ids":[]}, ...]}, ...]


In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # do this ONCE per kernel before using mp

from vllm_reward2 import build_llm, search

device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)

#device_groups = [[0], [1]]  # 2 workers on GPU 0 and 1

rm = build_llm(
    model_name=model_name,
    tensor_parallel_size=len(device_groups[0]),
    num_instances=len(device_groups),
    device_groups=device_groups,
    max_model_len=2500,
    max_num_seqs=64,
    gpu_memory_utilization=0.90,
    runner="pooling",
)

tokenizer = rm.tokenizer


In [ ]:
# Put before importing vllm
import os
#os.environ.setdefault("VLLM_CONFIGURE_LOGGING", "0")  # don't let vLLM set up its own handlers
#os.environ.setdefault("VLLM_LOGGING_LEVEL", "ERROR")  # lower verbosity if vLLM config still applies

# After imports, as a belt-and-suspenders:
import logging
logging.getLogger("vllm").setLevel(logging.ERROR)


def llm_template_func(row):
    query = row["query"]
    key = row["key"]
    message = [{'role': 'user', 'content':f"""Give me relevance score between\n\nQuery:{query}\n\nand\n\nSentence:{key}"""}]
    if len(message[0]["content"]) > 4000: message[0]["content"] = message[0]["content"][:4000]+"..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    #prompt = prompt[17:]
    return prompt
    

def search_key(queries, keys, tag2key, k_tag=2, k_key=5, num_instances = 1):
    # queries: ["...", ...]
    # keys: ["...", ...]
    # tag2key: [{"tag":"", "key_ids":[0,2, ...], "children":[{"key_ids":[2, ...]},{"key_ids":[0, ...]}, ...]}, ...]
    
    query2tag_ids = [{"tag_ids":[[] for _ in range(k_tag)]} for i in range(len(queries))]   # query2tag_ids = [{"tag_ids":[[]]}, ...]   # (num_queries, "tag_ids":(k_tag, depth))
    
    tags = [tag2key_dict["tag"] for tag2key_dict in tag2key]
    tags_request = [{"tags":[tags]} for _ in range(len(queries))]    # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))
    while_end = False
    depth = 0

    def get_tag_dict(tag_ids, tag_dict):
        if tag_ids == []: return None
    
        for tag_id in tag_ids[:-1]:
            if "children" not in tag_dict[tag_id]:
                return None
            else:
                tag_dict = tag_dict[tag_id]["children"]
        
        return tag_dict[tag_ids[-1]]
        
    '''
    def get_tag_dict(tag_ids, tag_dict):
        if tag_ids == []: return None
    
        for tag_id in tag_ids[:-1]:
            tag_dict = tag_dict[tag_id]["children"]
        
        return tag_dict[tag_ids[-1]]
    '''

    while not while_end:

        depth += 1

        requests = []
        query_and_n_top_ids = [] # [(query_id, n_top), ...] # (num_request)
        total_requests = 0

        for query_id in range(len(tags_request)):
            for nth_tag_ids, tag_list in enumerate(tags_request[query_id]["tags"]):
                query_and_n_top_ids.append((query_id, nth_tag_ids))
                requests.append({"query":queries[query_id], "keys":tag_list, "k": k_tag, "return_relevance":True})
                total_requests += len(tag_list)

        path = f"search_key-output{depth}-test4.json"
        if not os.path.exists(path):
            batch_size = total_requests // num_instances
            print(f"Graph Depth: {depth},  total_requests: {total_requests},  Batch size: {batch_size}")
            output = search(rm, requests, llm_template_func, topk = k_tag, batch_size=5000, timeout_s=4000)
    
            with open(path, "w") as f:
                json.dump(output, f)
        else:
            with open(path) as f:
                output = json.load(f)
        
        tags_request = [{"tags":[]} for _ in range(len(queries))] # tags_request = [{"tags":[[]]}, ...]   # (num_queries, "tags":(k_tag, num_tags_in_branch))
        result1 = {query_id:{"tag_ids_list":[], "relevance_list":[]} for query_id in range(len(queries))}   # {"query_id":{"tag_ids_list":[[3,1],],"relevance_list":[]}}
        for reuqest_id, output_dict in enumerate(output):
            query_id, nth_tag_ids = query_and_n_top_ids[reuqest_id]
            tags = []
            tag_relevance = []
            pre_tag_ids = query2tag_ids[query_id]["tag_ids"][nth_tag_ids]

            for top_nth in range(k_tag):
                try:  # output[query_id]["keys"][top_nth] can be index out of range. It goes to next output_dict.
                    new_tag_id = output_dict["keys"][top_nth]["key_id"]
                    relevance = output_dict["keys"][top_nth]["relevance"]
                    result1[query_id]["tag_ids_list"].append(pre_tag_ids+[new_tag_id])
                    result1[query_id]["relevance_list"].append(relevance)
                except:
                    continue

        while_end = True

        for query_id in result1:
            tag_relevance = result1[query_id]["relevance_list"]
            tag_ids_list = result1[query_id]["tag_ids_list"]
            if len(tag_relevance) == 0:
                top_tag_ids_list = []
            if len(tag_relevance) < k_tag:  # if there are not enough tag_ids
                _, indices = torch.topk(torch.tensor(tag_relevance), k=len(tag_relevance))
                top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]
            else:  # normal pattern
                _, indices = torch.topk(torch.tensor(tag_relevance), k=k_tag)
                top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]
                
            #_, indices = torch.topk(torch.tensor(tag_relevance), k=k_tag)
            #top_tag_ids_list = [tag_ids_list[index.item()] for index in indices]

            new_tag_ids_list = []
            for tag_ids in top_tag_ids_list:
                tag_dict = get_tag_dict(tag_ids, tag2key)
                if "children" not in tag_dict:
                    continue
                elif tag_dict["children"] == []:
                    continue

                tags = []
                for j in range(len(tag_dict["children"])):
                    tag = tag_dict["children"][j]["tag"]
                    tags.append(tag)

                while_end = False
                tags_request[query_id]["tags"].append(tags)
                new_tag_ids_list.append(tag_ids)

            query2tag_ids[query_id]["tag_ids"] = new_tag_ids_list
            
    query2key_ids = [] # [{"key_ids":[]}, ...]   # (num_queries, "key_ids":(n_total_hit_keys))
    total_requests = 0
    requests = []
    for query_id, _ in enumerate(query2tag_ids):
        combined_key_ids = []
        for tag_ids in _["tag_ids"]:
            tag_dict = get_tag_dict(tag_ids, tag2key)
            if "query_ids" in tag_dict:
                key_ids = tag_dict["query_ids"]
                combined_key_ids += key_ids
            
        
        query2key_ids.append({"key_ids":combined_key_ids})
        selected_keys = []
        for key_id in combined_key_ids:
            try:
                selected_keys.append(keys[key_id])
            except:
                continue
            #selected_keys = [keys[key_id] for key_id in combined_key_ids]
            
        requests.append({"query":queries[query_id], "keys":selected_keys, "k": k_key, "return_relevance":True})
        total_requests += len(selected_keys)

    #os.environ.setdefault("VLLM_CONFIGURE_LOGGING", "0")  # don't let vLLM set up its own handlers
    #os.environ.setdefault("VLLM_LOGGING_LEVEL", "ERROR")  # lower verbosity if vLLM config still applies

    batch_size = 10 # total_requests // num_instances
    print(f"Final Search,  total_requests: {total_requests},  Batch size: {batch_size}")
    output = search(rm, requests, llm_template_func, topk = k_key, batch_size=batch_size,  timeout_s=10000)

    # output : [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]
    return output

sentences = []
for i in range(len(df)):
    key = f"""{df.iloc[i]['text']}"""
    sentences.append(key)

questions = []
correct_ids = []
for id in range(len(df)):
    questions += query_dict[str(id)]["questions"]
    correct_ids += [id for _ in range(len(query_dict[str(id)]["questions"]))]

    
output = search_key(questions, sentences, tag_dict, k_tag=2, k_key=10, num_instances= num_instances)
# e.g. [{"query":, "query_id":0, "keys":[{"key_id":2,"key":,}, ...]}, ...]

for i in range(len(output)):
    output[i]["correct_id"] = correct_ids[i]
    for j in range(len(output[i]["keys"])):
        output[i]["keys"][j]["relevant_id"] = output[i]["keys"][j]["key_id"]

# output : [{"query":questions[0], "query_id":0, "keys":[{"relevant_id":8,"key":sentences[8],}, ...]}, ...]
with open(output_path, "w") as f:
    json.dump(output, f)

print("file_saved")


## Without Graph (Calculate all possible relevances between queries and keys)

In [ ]:
# Need to use vllm==0.10.1 version to use this model
!pip install -U vllm==0.10.1

In [ ]:
# Data Loading

working_dir = "/workspace/RMS_exp"
#exp_dir = f"{working_dir}/exp2"
model_name = f"/workspace/llama3b-rm-converted-model"
exp_eval_dir = f"{model_name}-eval"
#dataset_load_path = f"{exp_dir}/dataset"
data_name = "smollm-corpus"
tensor_parallel_size = 1
num_instances = 4
output_path = f"{exp_eval_dir}/relevance_dict.json"

import os
import pandas as pd
import json
import torch

if not os.path.exists(exp_eval_dir): os.makedirs(exp_eval_dir)

df = pd.read_csv(f"{working_dir}/data/{data_name}/df_small.csv")                            # df["text"] = ["sentence1", .... ]
df = df[:1000]
with open(f"{working_dir}/data/{data_name}/query_dict.json") as f:
    query_dict = json.load(f)                                                               # { "0": {"titles":["",...], "keywords":["",...], "questions":["",...], "irr_questions":["",...] } }
with open(f"{working_dir}/data/{data_name}/sentences_relevant_to_questions.json") as f:
    relevant_sentences = json.load(f)                                                       # [{"query":, "query_id":, "correct_id":, "keys":[{"key":, "key_id":,}, ...]}, ... ]
with open(f"{working_dir}/data/{data_name}/tag2query.json") as f:
    tag_dict = json.load(f)                                                                 # [{"tag":"", "query_ids":[], "children":[{"tag":"", "query_ids":[]}, ...]}, ...]


In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # do this ONCE per kernel before using mp

from vllm_reward2 import build_llm, search

device_groups = []
device_id = 0
for dp in range(num_instances):
    device_group = []
    for tp in range(tensor_parallel_size):
        device_group.append(device_id)
        device_id += 1

    device_groups.append(device_group)

#device_groups = [[0], [1]]  # 2 workers on GPU 0 and 1

rm = build_llm(
    model_name=model_name,
    tensor_parallel_size=len(device_groups[0]),
    num_instances=len(device_groups),
    device_groups=device_groups,
    max_model_len=2500,
    max_num_seqs=64,
    gpu_memory_utilization=0.90,
    runner="pooling",
)

tokenizer = rm.tokenizer


In [ ]:
# Build a full relevance_dict by scoring every query-key pair without graph pruning.
import logging

logging.getLogger("vllm").setLevel(logging.ERROR)


def llm_template_func(row):
    query = row["query"]
    key = row["key"]
    message = [{'role': 'user', 'content': f"Give me relevance score between\n\nQuery:{query}\n\nSentence:{key}"}]
    if len(message[0]["content"]) > 4000:
        message[0]["content"] = message[0]["content"][:4000] + "..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    return prompt


sentences = [str(df.iloc[i]["text"]) for i in range(len(df))]
queries = []
correct_ids = []
for idx in range(len(df)):
    question_list = query_dict[str(idx)]["questions"]
    queries.extend(question_list)
    correct_ids.extend([idx] * len(question_list))

num_queries = len(queries)
num_keys = len(sentences)
if num_queries == 0 or num_keys == 0:
    raise ValueError("No queries or keys available to score.")

query_batch_size = 20
encode_batch_size = 10
relevance_dict = []
batch_on = True

if batch_on:

    for start in range(0, num_queries, query_batch_size):
        batch_queries = queries[start:start + query_batch_size]
        requests = [
            {"query": query, "keys": sentences, "return_relevance": True}
            for query in batch_queries
        ]
        outputs = search(
            rm,
            requests,
            llm_template_func,
            topk=num_keys,
            batch_size=encode_batch_size,
            timeout_s=10000,
        )
        for local_idx, result in enumerate(outputs):
            global_idx = start + local_idx
            result["query_id"] = global_idx
            result["correct_id"] = correct_ids[global_idx]
            for key_item in result["keys"]:
                key_item["relevant_id"] = key_item["key_id"]
            relevance_dict.append(result)
        print(f"Processed {min(start + query_batch_size, num_queries)} / {num_queries} queries", flush=True)

else:
    requests = [
        {"query": query, "keys": sentences, "return_relevance": True}
        for query in queries
    ]
    outputs = search(
        rm,
        requests,
        llm_template_func,
        topk=num_keys,
        batch_size=encode_batch_size,
        timeout_s=10000,
    )
    for global_idx, result in enumerate(outputs):
        result["query_id"] = global_idx
        result["correct_id"] = correct_ids[global_idx]
        for key_item in result["keys"]:
            key_item["relevant_id"] = key_item["key_id"]
        relevance_dict.append(result)

full_output_path = f"{exp_eval_dir}/relevance_dict_without_graph.json"
with open(full_output_path, "w") as f:
    json.dump(relevance_dict, f)

print(f"Wrote relevance scores for all query-key pairs to {full_output_path}")



## Derive nDCG

In [ ]:
!pip install ijson
!pip install pandas

In [ ]:
import json, ijson
import pandas as pd
import numpy as np
import time
import math

In [ ]:
def compute_precision(expected_ids: list[str], retrieved_ids: list[str]) -> float:
    """
    Computes precision.
    Args:
        expected_ids (List[str]): The ground truth node_ids.
        retrieved_ids (List[str]): The node_ids from retrieved chunks.
    Returns:
        float: Precision score as a decimal.
    """
    if not expected_ids:
        return 0.0 

    # Find the number of relevant documents that were retrieved
    relevant_and_retrieved = set(expected_ids).intersection(set(retrieved_ids))
    
    # Precision = (relevant retrieved) / (total retrieved)
    # return len(relevant_and_retrieved) / len(retrieved_ids)
    return len(retrieved_ids) / len(expected_ids)
def compute_DCG(rank_score):
    if rank_score == 0:
        DCG = 0
    else:
        DCG = 1/math.log((rank_score+1),2)
    return DCG

In [ ]:
#Problem with precisiob and recall definition
!pip install ijson

file_path = '/workspace/llama3b-rm-converted-model-eval/relevance_dict.json'

item_count = 0
items_to_print = 100
#10000 items need 3 seconds
#100000 items need 45 seconds

expected_id = []
retrieved_id = []
not_found_querie_ids = []
precision_result = [] 
rank_scores = []
MAP_score = 0
IDCG = 1
nDCG = 0
DCG = 0

try:
    with open(file_path, 'rb') as file:
        print(f"Reading first {items_to_print} items from '{file_path}'...")
        # 'item' refers to each element in the top-level array
        start_time = time.time()
        for item in ijson.items(file, 'item'):
            # The 'item' is a standard Python dictionary
            # print(f"--- Item {item_count + 1} ---")
            # Print a few key details to inspect the structure
            # if 'query' in item:
            #     print(f"query: {item['query']}")
            if 'query_id' in item:
                # print(f"query id: {item['query_id']}")
                found_match = False
            if 'correct_id' in item:
                # print(f"correct: {item['correct_id']}")
                c_id = int(item['correct_id'])
                expected_id.append(c_id)
                print(f"correct id: {expected_id}")
            if 'keys' in item:
                key_list=item["keys"]
                list_size = len(key_list)
                print(f"The keys have a total of {len(key_list)} items")
                # print(f"keys: {item['keys']}")
                for index, key in enumerate(key_list):
                    # print(i)
                    keyid=int(key["key_id"])
                    print(f"this is the id for this key: {keyid}")
                    # print(f"this is the id for the correct id: {c_id}")
                    if keyid == c_id:
                        retrieved_id.append(keyid)
                        found_match = True

                        position = index + 1
                        # score = (6-position) / 5
                        score = (list_size+1-position)/list_size
                        rank_scores.append(score)
                        print(f"retrieved ids right now is: {retrieved_id}")
                        break
                    # else:
                    #     incorrect_id.append(keyid)
                    #     print(f"Incorrect ids right now is: {incorrect_id}")
                    # print("----------------------------------------")
                #Each item have a query, query id, and key list that contain 5 keys 
                # if the correct key id appear in this top ten we add the id into the retrieved id, if not we add zero
            if not found_match:
                not_found_querie_ids.append(item['query_id'])
                score = 0
                rank_scores.append(score)

            ##Calculate nDCG
            print(f"score: {score}")
            if score == 0:
                rel = 0
            else:
                # rel = 6-5*score
                rel = (list_size+1)-(list_size)*score
            DCG = compute_DCG(rel)
            print(f"DCG: {DCG}")
            nDCG = nDCG + DCG/IDCG/items_to_print
            
            ##Calculate Precision
            # print(expected_id)
            # print(retrieved_id)
            # precision = compute_precision(expected_ids=expected_id, retrieved_ids=retrieved_id)
            # precision_result.append(precision)
            # print(f"precision: {precision}")
            # MAP_score = MAP_score + precision*score/items_to_print
            # print(f"MAP: {MAP_score}")
            
            item_count += 1
            if item_count >= items_to_print:
                break
        elapsed_time = time.time() - start_time
        print(f"✅ Process completed in {elapsed_time:.2f} seconds.")
        # array2D = np.array([precision_result])
        # mean_results = np.mean(array2D, axis=1)
        # results_df = pd.DataFrame(mean_results)
        # results_df.index = ["Precision@10"]  

        print(expected_id)
        print(retrieved_id)
        print(f"The length of expected list is {len(expected_id)}")
        print(f"The length of retrieved list is {len(retrieved_id)}")
        print(f"The length of incorrect list of query id is {len(not_found_querie_ids)}")
        # print(f"The scores are {rank_scores}")
        # print(f"The MAP score is {MAP_score}")
        print(f"The nDCG score is {nDCG}")
        # print(results_df)
        
except FileNotFoundError:
    print(f"Error: The file at '{file_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## Old but can be relevant

In [ ]:
import torch

with open(test_problem_and_answer_save_path) as f:
    data_dict = json.load(f)
    problems = data_dict["problems"]
    answers = data_dict["answers"]

with open(train_data_dict_save_path) as f:
    train_data_dict = json.load(f)
    train_advices = train_data_dict["advice"]

with open(test_data_dict_save_path) as f:
    test_data_dict = json.load(f)
    test_advices = test_data_dict["advice"]

if overlapping_advice_data_load_path:
    with open(overlapping_advice_data_load_path) as f:
        overlapping_advice_data = json.load(f)

    accepted_ids = overlapping_advice_data["accepted_ids"]

    accepted_train_advices = []
    for a_id in accepted_ids:
        accepted_train_advices.append(train_advices[a_id])

    n_train_advices = len(accepted_train_advices)
    advices = accepted_train_advices+test_advices
    relevance = get_relevance(problems, answers, advices)
    
    train_relevance = relevance[:,:n_train_advices]
    test_relevance = relevance[:,n_train_advices:]
    
    torch.save(train_relevance, train_relevance_save_path)
    torch.save(test_relevance, test_relevance_save_path)
    
else:
    n_train_advices = len(train_advices)
    advices = train_advices+test_advices
        
    relevance = get_relevance(problems, answers, advices)
    
    train_relevance = relevance[:,:n_train_advices]
    test_relevance = relevance[:,n_train_advices:]
    
    torch.save(train_relevance, train_relevance_save_path)
    torch.save(test_relevance, test_relevance_save_path)

import json
import torch

with open(train_data_dict_save_path) as f:
    train_data_dict = json.load(f)
    train_advices = train_data_dict["advice"]
    train_advice_problem_ids = train_data_dict["problem_id"]  # index of train_advices -> original_problem_id where advice belongs

with open(test_problem_and_answer_save_path) as f:
    data_dict = json.load(f)
    test_problems = data_dict["problems"]
    test_answers = data_dict["answers"]

with open(test_data_dict_save_path) as f:
    test_data_dict = json.load(f)
    test_advices = test_data_dict["advice"]
    test_advice_problem_ids = test_data_dict["problem_id"]  # index of test_advices -> original_problem_id where advice belongs

train_relevance = torch.load(train_relevance_save_path)  # [len(test_problems), len(train_advices)]
test_relevance = torch.load(test_relevance_save_path)  # [len(test_problems), len(test_dvices)]
n_test_problems, n_train_advices = train_relevance.shape
n_test_problems2, n_test_advices = test_relevance.shape
if not n_test_problems == n_test_problems2: raise Exception("wrong shape")
relevance = torch.cat((train_relevance, test_relevance), dim=1)

advice_problem_ids = train_advice_problem_ids + test_advice_problem_ids

from datasets import load_from_disk
formatted_dataset = load_from_disk(dataset_load_path)
train_problem_ids = formatted_dataset["train"]["problem_id"]
test_problem_ids = formatted_dataset["test"]["problem_id"]


num_correct_advice_dict = {str(k):[] for k in k_values}
for new_problem_id, original_problem_id in enumerate(test_problem_ids):
    top_values, top_indices = torch.topk(relevance[new_problem_id], k=num_retrieval) # [num_retrieval]
    #print(top_values, top_indices)
    top_indices = top_indices.tolist()
    
    for k in k_values:
        top_indices_ = top_indices[:k]
        top_original_problem_ids = torch.tensor([advice_problem_ids[top_index] for top_index in top_indices_])
        #print(top_original_problem_ids, original_problem_id)
        result_tensor = torch.where(top_original_problem_ids==original_problem_id, 1, 0)
        n_correct_advice = torch.sum(result_tensor).item()
        num_correct_advice_dict[str(k)].append(n_correct_advice)

eval_result = {"num_correct_advice_dict":num_correct_advice_dict, "mean_n_correct_advice":{}}

for k in k_values:
    print()
    print(f"top-{k}")
    mean_n_correct_advice = sum(num_correct_advice_dict[str(k)])/len(num_correct_advice_dict[str(k)])
    print(f"mean n_correct_advice: {mean_n_correct_advice}")
    eval_result["mean_n_correct_advice"][str(k)] = mean_n_correct_advice

with open(eval_result_save_path, "w") as f:
    json.dump(eval_result, f)

print("eval_result has been saved")


### Old

In [ ]:
from typing import Any, Dict, List
import numpy as np
import ray
from packaging.version import Version
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy
from vllm import LLM, SamplingParams
assert Version(ray.__version__) >= Version(
    "2.22.0"), "Ray version must be at least 2.22.0"
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM

class pooler_config:
    def __init__(self):
        self.pooling_type = "LAST"
        self.normalize = False
        self.softmax = False
        self.softmax = False
        self.step_tag_id = None
        self.returned_token_ids = None

pooler_config_ = pooler_config()

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left",add_eos_token=True,add_bos_token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    config.pad_token_id = config.eos_token_id


# Create a class to do batch inference.
class LLMPredictor:

    def __init__(self):
        # Create an LLM.
        self.llm = LLM(model=model_name,
                       dtype="bfloat16",
                       #dtype="float32",
                       tensor_parallel_size=tensor_parallel_size,
                       task="embed",
                       override_pooler_config=pooler_config_,)

    def __call__(self, batch: Dict[str, np.ndarray]) -> Dict[str, list]:
        outputs = self.llm.embed(batch["prompt"])

        embeddings = []
        for output in outputs:
            embeddings.append(output.outputs.embedding)
            #generated_text.append(' '.join([o.text for o in output.outputs]))

        return {
            "embeddings": embeddings
        }


# Read one text file from S3. Ray Data supports reading multiple files
# from cloud storage (such as JSONL, Parquet, CSV, binary format).
#ds = ray.data.read_text("s3://anonymous@air-example-data/prompts.txt")

# For tensor_parallel_size > 1, we need to create placement groups for vLLM
# to use. Every actor has to have its own placement group.
def scheduling_strategy_fn():
    # One bundle per tensor parallel worker
    pg = ray.util.placement_group(
        [{
            "GPU": 1,
            "CPU": 1
        }] * tensor_parallel_size,
        strategy="STRICT_PACK",
    )
    return dict(scheduling_strategy=PlacementGroupSchedulingStrategy(
        pg, placement_group_capture_child_tasks=True))


resources_kwarg: Dict[str, Any] = {}
if tensor_parallel_size == 1:
    # For tensor_parallel_size == 1, we simply set num_gpus=1.
    resources_kwarg["num_gpus"] = 1
else:
    # Otherwise, we have to set num_gpus=0 and provide
    # a function that will create a placement group for
    # each instance.
    resources_kwarg["num_gpus"] = 0
    resources_kwarg["ray_remote_args_fn"] = scheduling_strategy_fn



def get_relevance(problems, answers, advices):
    
    import pandas as pd
    import itertools
    
    # Generate the Cartesian product
    new_problem_ids = list(range(len(problems)))
    combinations = list(itertools.product(new_problem_ids, advices))
    df = pd.DataFrame(combinations, columns=['problem_id', 'advice'])
    df['problem'] = df['problem_id'].apply(lambda idx: problems[idx])
    df['answer'] = df['problem_id'].apply(lambda idx: answers[idx])
    
    from datasets import Dataset
    dataset1 = Dataset.from_pandas(df)
    
    def format(row):
        advice = row["advice"]
        problem = row["problem"]
        answer = row["answer"]
        answer = answer[:4000]
        
        message = [
          {'role': 'user', 'content': f"Give me an advice to the problem and answer below;\n\nProblem:{problem}\n\nAnswer:{answer}"},
          {'role': 'assistant', 'content': f"{advice}"}
        ]
        prompt = tokenizer.apply_chat_template(message, tokenize=False)
        prompt = prompt[17:]  # to eliminate <|begin_of_text|> because vllm automatically add it to prompt
        row["prompt"] = prompt
        return row
    
    formatted_dataset = dataset1.map(format)
    df_formatted = formatted_dataset.to_pandas()
    list_of_prompts = df_formatted[['prompt']].to_dict('records')  # [{"prompt":".."}, ...]
    
    ds = ray.data.from_items(list_of_prompts)
    #ds = ray.data.read_text("s3://anonymous@air-example-data/prompts.txt")
    
    batch_size = len(list_of_prompts)//num_instances
    
    # Apply batch inference for all input data.
    ds = ds.map_batches(
        LLMPredictor,
        # Set the concurrency to the number of LLM instances.
        concurrency=num_instances,
        # Specify the batch size for inference.
        batch_size=batch_size,
        **resources_kwarg,
    )
    
    # Peek first 10 results.
    # NOTE: This is for local testing and debugging. For production use case,
    # one should write full result out as shown below.
    # outputs = ds.take(limit=10)
    outputs = ds.take_all()  # [{"embeddings":(2d list)}, ...]
    
    #outputs = model.embed(prompts)
    from datasets import Dataset
    import torch
    dataset1 = Dataset.from_list(outputs)
    print("Putting All Embeddings onto GPU...")
    all_embeddings = torch.tensor(dataset1["embeddings"]).to("cuda")
    #all_embeddings = torch.tensor([outputs[i].outputs.embedding for i in range(len(outputs))]).to("cuda")
    rm_head = torch.load(score_path, weights_only=True)
    rm_head = rm_head.to(all_embeddings.device)
    print("Calculating matmul...")
    advice_rewards = torch.matmul(all_embeddings, rm_head.transpose(0,1)).squeeze()
    relevance = advice_rewards.reshape(len(problems), len(advices))  # [len(problems), len(advices)]
    print("Obtained Relevance")
    
    return relevance

#top_advice_rewards, advice_indices = torch.topk(reshaped_advice_rewards, k=num_retrival) # [len(problems), num_advices]
#torch.save(advice_indices, retrieved_advice_ids_save_path)  #all_advice_indices: [len(test_problem_ids), num_retrival]


In [ ]:
import torch

with open(test_problem_and_answer_save_path) as f:
    data_dict = json.load(f)
    problems = data_dict["problems"]
    answers = data_dict["answers"]

with open(train_data_dict_save_path) as f:
    train_data_dict = json.load(f)
    train_advices = train_data_dict["advice"]

with open(test_data_dict_save_path) as f:
    test_data_dict = json.load(f)
    test_advices = test_data_dict["advice"]

if overlapping_advice_data_load_path:
    with open(overlapping_advice_data_load_path) as f:
        overlapping_advice_data = json.load(f)

    accepted_ids = overlapping_advice_data["accepted_ids"]

    accepted_train_advices = []
    for a_id in accepted_ids:
        accepted_train_advices.append(train_advices[a_id])

    n_train_advices = len(accepted_train_advices)
    advices = accepted_train_advices+test_advices
    relevance = get_relevance(problems, answers, advices)
    
    train_relevance = relevance[:,:n_train_advices]
    test_relevance = relevance[:,n_train_advices:]
    
    torch.save(train_relevance, train_relevance_save_path)
    torch.save(test_relevance, test_relevance_save_path)
    
else:
    n_train_advices = len(train_advices)
    advices = train_advices+test_advices
        
    relevance = get_relevance(problems, answers, advices)
    
    train_relevance = relevance[:,:n_train_advices]
    test_relevance = relevance[:,n_train_advices:]
    
    torch.save(train_relevance, train_relevance_save_path)
    torch.save(test_relevance, test_relevance_save_path)

In [ ]:
import json
import torch

with open(train_data_dict_save_path) as f:
    train_data_dict = json.load(f)
    train_advices = train_data_dict["advice"]
    train_advice_problem_ids = train_data_dict["problem_id"]  # index of train_advices -> original_problem_id where advice belongs

with open(test_problem_and_answer_save_path) as f:
    data_dict = json.load(f)
    test_problems = data_dict["problems"]
    test_answers = data_dict["answers"]

with open(test_data_dict_save_path) as f:
    test_data_dict = json.load(f)
    test_advices = test_data_dict["advice"]
    test_advice_problem_ids = test_data_dict["problem_id"]  # index of test_advices -> original_problem_id where advice belongs

train_relevance = torch.load(train_relevance_save_path)  # [len(test_problems), len(train_advices)]
test_relevance = torch.load(test_relevance_save_path)  # [len(test_problems), len(test_dvices)]
n_test_problems, n_train_advices = train_relevance.shape
n_test_problems2, n_test_advices = test_relevance.shape
if not n_test_problems == n_test_problems2: raise Exception("wrong shape")
relevance = torch.cat((train_relevance, test_relevance), dim=1)

advice_problem_ids = train_advice_problem_ids + test_advice_problem_ids

from datasets import load_from_disk
formatted_dataset = load_from_disk(dataset_load_path)
train_problem_ids = formatted_dataset["train"]["problem_id"]
test_problem_ids = formatted_dataset["test"]["problem_id"]


num_correct_advice_dict = {str(k):[] for k in k_values}
for new_problem_id, original_problem_id in enumerate(test_problem_ids):
    top_values, top_indices = torch.topk(relevance[new_problem_id], k=num_retrieval) # [num_retrieval]
    #print(top_values, top_indices)
    top_indices = top_indices.tolist()
    
    for k in k_values:
        top_indices_ = top_indices[:k]
        top_original_problem_ids = torch.tensor([advice_problem_ids[top_index] for top_index in top_indices_])
        #print(top_original_problem_ids, original_problem_id)
        result_tensor = torch.where(top_original_problem_ids==original_problem_id, 1, 0)
        n_correct_advice = torch.sum(result_tensor).item()
        num_correct_advice_dict[str(k)].append(n_correct_advice)

eval_result = {"num_correct_advice_dict":num_correct_advice_dict, "mean_n_correct_advice":{}}

for k in k_values:
    print()
    print(f"top-{k}")
    mean_n_correct_advice = sum(num_correct_advice_dict[str(k)])/len(num_correct_advice_dict[str(k)])
    print(f"mean n_correct_advice: {mean_n_correct_advice}")
    eval_result["mean_n_correct_advice"][str(k)] = mean_n_correct_advice

with open(eval_result_save_path, "w") as f:
    json.dump(eval_result, f)

print("eval_result has been saved")


# Evaluation (?)

In [ ]:
# reset
import json, os, re
with open("file_dict.json") as f:
    file_dict = json.load(f)

for file_path in file_dict:
    with open(file_path, "w") as f:
        f.write(file_dict[file_path])

In [ ]:
import json
with open("data_dict.json") as f:
    data_dict = json.load(f)

In [ ]:
id = 1
bug = data_dict["bug"][id]
file_path = data_dict["file_path"][id]
line_list = data_dict["line_list"][id]
code_lines_list = data_dict["code_lines_list"][id]
print(file_path)
print(bug)

In [ ]:
# modify
if type(line_list)==list:
    line = line_list[0]
elif type(line_list)==int:
    line = line_list
else:
    raise Exception("error")

code_lines_list[line] = "//" + code_lines_list[line]
modified_code = "\n".join(code_lines_list)

with open(file_path, "w") as f:
    json.dump(modified_code, f)
    

In [ ]:
# for normal openhands
print(f'''You are an expert software engineer who writes concise, syntactically correct git patches.  
Output **only** one unified `git diff` that can be applied with `git apply` or `patch`.  
Do **not** include explanations, commentary, or extra text.

Project context:
This code is a script for car racing game created by unity. 

Bug description:
{bug}

Generate a patch that:
- Keeps code style consistent with surrounding code.
- Introduces no unrelated changes.
- Ensures all tests pass (and the bug is fixed).

Also prepend a Conventional Commits-style commit message above the diff.''')


In [ ]:
# for rmsearch + openhands
print(f'''You are an expert software engineer who writes concise, syntactically correct git patches.  
Output **only** one unified `git diff` that can be applied with `git apply` or `patch`.  
Do **not** include explanations, commentary, or extra text.
Also, when you search some files which may contain some error, try to use rmsearch as a tool to get relevant file paths.

Project context:
This code is a script for car racing game created by unity. 

Bug description:
{bug}

Generate a patch that:
- Keeps code style consistent with surrounding code.
- Introduces no unrelated changes.
- Ensures all tests pass (and the bug is fixed).

Also prepend a Conventional Commits-style commit message above the diff.''')

In [ ]:
print(f'''You are an expert software engineer who writes concise, syntactically correct git patches.  
Output **only** one unified `git diff` that can be applied with `git apply` or `patch`.  
Do **not** include explanations, commentary, or extra text.
Also, when you search some files which may contain some error, try to use rmsearch as a tool to get relevant file paths.

Project context:
This code is a script for car racing game created by unity. 

Bug description:
Player car and AI car don't move at all.

Generate a patch that:
- Keeps code style consistent with surrounding code.
- Introduces no unrelated changes.
- Ensures all tests pass (and the bug is fixed).

Also prepend a Conventional Commits-style commit message above the diff.''')

In [ ]:
def same_except_eol(text1: str, text2: str) -> bool:
    """
    Return True if the two pieces of text are identical after normalizing
    Windows (CR-LF) and old-Mac (CR) line endings to Unix (LF).
    """
    def _normalize(s: str) -> str:
        # 1. Convert CR-LF → LF
        s = s.replace("\r\n", "\n")
        # 2. Convert any lone CR → LF
        s = s.replace("\r", "\n")
        return s

    return _normalize(text1) == _normalize(text2)

same_except_eol

# Evaluation (Huan Eval)

In [ ]:
"""
Run the server with:

pip3 install /workspace/RMSearch fastapi uvicorn[standard]
uvicorn main:app --host 0.0.0.0 --port 8000
"""

from typing import List, Optional
from pydantic import BaseModel
from rmsearch import Search
import json

# ── Search engine instance ────────────────────────────────────────────────────
search_engine = Search(
    model_name="/workspace/racing_debug/exp3-model1-step120-converted-model",
    tensor_parallel_size=1,
    pipeline_parallel_size=1,
)

tokenizer = search_engine.tokenizer

def llm_template_func(query, key):
    message = [
      {'role': 'user', 'content': f"Give me an expected bug issue to the file content;\n\nFile Content:{key}"},
      {'role': 'assistant', 'content': f"{query}"}
    ]
    message[0]["content"] = message[0]["content"][:4000]+"..."
    prompt = tokenizer.apply_chat_template(message, tokenize=False)
    #prompt = prompt[17:]
    return prompt

search_engine.llm_template = llm_template_func

# ── Fallback keys (only used when the caller omits "keys") ───────────────────
with open("file_dict_bug.json") as f:
    file_dict = json.load(f)

code_list = []
file_paths = []
for file_path in file_dict:
    code = file_dict[file_path]
    code_list.append(code)
    file_paths.append(file_path)
    
DEFAULT_KEYS: List[str] = code_list

# ── Request / Response schemas ───────────────────────────────────────────────
class SearchRequest(BaseModel):
    queries: List[str]
    keys: Optional[List[str]] = None        # optional
    k: Optional[int] = 5

class KeyOut2(BaseModel):
    key_id: int
    code: str
    file_path: str

class QueryOut2(BaseModel):
    query: str
    query_id: int
    keys: List[KeyOut2]

async def rmsearch2(req: SearchRequest):
    """
    Body examples
    -------------
    1) Provide queries only (use server-side default keys):
        {
          "queries": ["How to make LLM?", "What's the capital of Japan?"]
        }

    2) Provide both queries and custom keys:
        {
          "queries": ["How to make LLM?"],
          "keys":    ["LLM is Large Language Model ..."]
        }
    """
    
    keys = req.keys or DEFAULT_KEYS                    # choose key set
    output = await search_engine(req.queries, keys, k=req.k, return_relevance=True)  # e.g. [{"query":, "query_id":0, "keys":[{"key_id":8,"key":,}, ...]}, ...]

    #print(output)
    # Build the required structure
    response: List[QueryOut2] = []
    for output_dict in output:
        key_objs = [KeyOut2(key_id=key_dict["key_id"], code=key_dict["key"], file_path=file_paths[key_dict["key_id"]]) for key_dict in output_dict["keys"]]
        response.append(QueryOut2(query=output_dict["query"], query_id=output_dict["query_id"], keys=key_objs))

    return response

In [ ]:
output = await rmsearch2(SearchRequest(queries=["The AI cars don't move."], k=10))

In [ ]:
for i in range(10):
    print(file_paths[output[0].keys[i].key_id])

# Expand Dataset

In [ ]:
# Use Model from local environment

import asyncio
from transformers import AutoTokenizer
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams
import time, warnings
import os

model_name = "/workspace/qwen7b"
tensor_parallel_size = 1

engine_args = AsyncEngineArgs(
    model = model_name,
    tensor_parallel_size = tensor_parallel_size,
    gpu_memory_utilization=0.95,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')



In [ ]:
file_paths = ["/Users/multivac/openhands/racing_test_bug/Assets/Racing Starter Kit/RSK Assets/Scripts/AI/AICarAvoidanceSmartSteering.cs",
                "/Users/multivac/openhands/racing_test_bug/Assets/Racing Starter Kit/RSK Assets/Scripts/Car/CarAIControl.cs"]

import json
with open("file_dict_bug.json") as f:
    file_dict = json.load(f)

system = f"""You are an excellent code assistant. Judge which file is more relevant to fix the issue, file 1 or file 2. Return the id of more relevant file following the designated output format."""
    
prompt = f"""Issue:
AI cars don't move.

File 1:
```{file_paths[0]}
{file_dict[file_paths[0]]}
```

File 2:
```{file_paths[1]}
{file_dict[file_paths[1]]}
```

Output Format:
'''
(Provide your thought first)

<ID>1 or 2</ID>
'''

Let's think step by step."""

prompt = tokenizer.apply_chat_template(
                [{"role": "system", "content":system}, {"role": "user", "content":prompt}],
                tokenize=False,
                add_generation_prompt=True
            )

print(prompt)

final_output = None
results_generator = engine.generate(prompt, SamplingParams(temperature=0, max_tokens=9000), 0)
async for request_output in results_generator:
    # print(request_output) => for streaming
    final_output = request_output

output = final_output.outputs[0].text

output

In [ ]:
import re
def extract_text(textC: str, textB: str) -> str | None:
    # Build a non-greedy pattern like r'<tag>(.*?)</tag>'
    pattern = rf"<{re.escape(textB)}>(.*?)</{re.escape(textB)}>"
    match = re.search(pattern, textC, flags=re.DOTALL)
    return match.group(1) if match else None

def extract_int(text: str) -> int | None:
    """
    Return the first integer found in `text`.
    If no digits appear, return None.
    """
    match = re.search(r"-?\d+", text)  # handles negative numbers too
    return int(match.group()) if match else None

In [ ]:
prompt = f"""Judge which file is more relevant to fix the issue? Return the id of more relevant file following the output format.

Issue:
AI cars don't move.

File 1:
```{file_paths[0]}
{file_dict[file_paths[0]]}
```

File 2:
```{file_paths[1]}
{file_dict[file_paths[1]]}
```

Output Format:
'''
<ID>1 or 2</ID>
'''
"""

final_output = None
results_generator = engine.generate(prompt, SamplingParams(temperature=0, max_tokens=9000), 0)
async for request_output in results_generator:
    # print(request_output) => for streaming
    final_output = request_output

output = final_output.outputs[0].text

output

In [ ]:
id = extract_text(output, "ID")
if not id: id = extract_int(output[-10:])
id

In [ ]:
for key in file_dict:
    print(key)

In [ ]:
file_paths = ["/Users/multivac/openhands/racing_test_bug/Assets/Racing Starter Kit/RSK Assets/Scripts/AI/AICarAvoidanceSmartSteering.cs",
                "/Users/multivac/openhands/racing_test_bug/Assets/Racing Starter Kit/RSK Assets/Scripts/PlayersSpawner.cs"]


system = f"""You are an excellent code assistant. Judge which file is more relevant to fix the issue, file 1 or file 2. Return the id of more relevant file following the designated output format."""
    
prompt = f"""Issue:
AI cars don't move.

File 1:
```{file_paths[0]}
{file_dict[file_paths[0]]}
```

File 2:
```{file_paths[1]}
{file_dict[file_paths[1]]}
```

Output Format:
'''
(Provide your step by step thought first)

<ID>1 or 2</ID>
'''

Let's think step by step."""

prompt = tokenizer.apply_chat_template(
                [{"role": "system", "content":system}, {"role": "user", "content":prompt}],
                tokenize=False,
                add_generation_prompt=True
            )

final_output = None
results_generator = engine.generate(prompt, SamplingParams(temperature=0, max_tokens=9000), 0)
async for request_output in results_generator:
    # print(request_output) => for streaming
    final_output = request_output

output = final_output.outputs[0].text

output

In [ ]:
id = extract_text(output, "ID")
if not id: id = extract_int(output[-10:])
id

In [ ]:
output